In [ ]:
# ============================================
# CELL 1: KILL PORT & INSTALL PACKAGES
# ============================================
!fuser -k 5000/tcp 2>/dev/null || true

!pip install -q flask flask-cors pyngrok ultralytics opencv-python-headless pillow google-generativeai openai fal-client requests

In [ ]:
# ============================================
# CELL 2: ALL IMPORTS
# ============================================
import cv2
import time
import numpy as np
from ultralytics import YOLO
import os
import json
import re
import io
import requests
import threading
from base64 import b64decode, b64encode
from PIL import Image

import google.generativeai as genai
from openai import OpenAI
import fal_client

from flask import Flask, request as flask_request, jsonify, render_template_string, session
from flask_cors import CORS
from pyngrok import ngrok as pyngrok_client

print("✅ All imports loaded!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ All imports loaded!


In [ ]:
# ============================================
# CELL 3: API CONFIGURATION
# ============================================

# --- OpenRouter (Chatbot + Fashion Recommendations) ---
chatbot_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=" "
)

# --- Gemini (Photo analysis) ---
GEMINI_API_KEY = " "
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("models/gemini-2.5-flash")

# --- fal.ai (Outfit Image Generation) ---
FAL_API_KEY = " "
os.environ["FAL_KEY"] = FAL_API_KEY

print("✅ API keys configured!")

✅ API keys configured!


In [ ]:
# ============================================
# CELL 4: CONFIGURATION CONSTANTS
# ============================================
INTENT_CONFIG = {
    "basic": {
        "timer_seconds": 3,
        "description": "Face detection for skin tone analysis",
        "required_parts": ["face"],
        "crop_to": "face"
    },
    "matching": {
        "timer_seconds": 5,
        "description": "Full body detection for outfit matching",
        "required_parts": ["face", "hips", "ankles"],
        "crop_to": "full_body"
    },
    "suggestion": {
        "timer_seconds": 4,
        "description": "Region-specific detection for outfit suggestions",
        "required_parts": None,
        "crop_to": None
    }
}

REGION_CONFIG = {
    "upper": {
        "required_parts": ["face", "hips"],
        "crop_to": "upper_body",
        "description": "Head to waist"
    },
    "lower": {
        "required_parts": ["hips", "ankles"],
        "crop_to": "lower_body",
        "description": "Hips to feet"
    }
}

DEFAULT_CONFIG = {
    "kp_conf_thresh": 0.3,
    "model_path": "yolov8s-pose.pt",
    "selection_mode": "largest",
    "max_retries": 5,
    "bbox_margin": 0.15,
    "camera_index": 0,
    "show_preview": True
}

KEYPOINTS = {
    "nose": 0, "left_eye": 1, "right_eye": 2,
    "left_shoulder": 5, "right_shoulder": 6,
    "left_hip": 11, "right_hip": 12,
    "left_knee": 13, "right_knee": 14,
    "left_ankle": 15, "right_ankle": 16
}

MAX_HISTORY = 30

print("✅ Config loaded!")

✅ Config loaded!


In [ ]:
# ============================================
# CELL 5: LOAD YOLO MODEL ONCE
# ============================================
print("⏳ Loading YOLO pose model...")
_yolo_model = YOLO(DEFAULT_CONFIG["model_path"])
print("✅ YOLO model loaded!")

⏳ Loading YOLO pose model...
✅ YOLO model loaded!


In [ ]:
# ============================================
# CELL 6: PROFESSIONAL UI CLASS
# ============================================
class ProfessionalUI:
    """Status tracking for web mode"""

    def __init__(self):
        self.status_logs = []

    def _get_time_str(self):
        return time.strftime("%H:%M:%S")

    def add_status(self, icon, message):
        self.status_logs.append({
            "time": self._get_time_str(),
            "icon": icon,
            "message": message
        })
        if len(self.status_logs) > 10:
            self.status_logs = self.status_logs[-10:]
        print(f"  {icon} {message}")

ui = ProfessionalUI()
print("✅ UI loaded!")

✅ UI loaded!


In [ ]:
# ============================================
# CELL 7: DETECTION HELPER FUNCTIONS
# ============================================
def _image_to_base64(image_bgr):
    _, buffer = cv2.imencode('.jpg', image_bgr)
    return b64encode(buffer).decode('utf-8')


def _check_face_visible(person_kp, conf_thresh):
    return (person_kp[KEYPOINTS["nose"]][2] > conf_thresh and
            (person_kp[KEYPOINTS["left_eye"]][2] > conf_thresh or
             person_kp[KEYPOINTS["right_eye"]][2] > conf_thresh))


def _check_hips_visible(person_kp, conf_thresh):
    return (person_kp[KEYPOINTS["left_hip"]][2] > conf_thresh or
            person_kp[KEYPOINTS["right_hip"]][2] > conf_thresh)


def _check_ankles_visible(person_kp, conf_thresh):
    return (person_kp[KEYPOINTS["left_ankle"]][2] > conf_thresh or
            person_kp[KEYPOINTS["right_ankle"]][2] > conf_thresh)


def _validate_required_parts(person_kp, required_parts, conf_thresh):
    checks = {
        "face": lambda: _check_face_visible(person_kp, conf_thresh),
        "hips": lambda: _check_hips_visible(person_kp, conf_thresh),
        "ankles": lambda: _check_ankles_visible(person_kp, conf_thresh)
    }
    return all(checks[part]() for part in required_parts if part in checks)


def _get_crop_bbox(person_kp, box, crop_to, conf_thresh=0.3):
    if crop_to == "face":
        points = [person_kp[KEYPOINTS[k]][:2]
                  for k in ["nose", "left_eye", "right_eye"]]
        valid = [p for p in points if p[0] > 0 and p[1] > 0]
        if not valid:
            return tuple(int(v) for v in box)
        x_coords = [p[0] for p in valid]
        y_coords = [p[1] for p in valid]
        cx = (min(x_coords) + max(x_coords)) // 2
        cy = (min(y_coords) + max(y_coords)) // 2
        size = max(max(x_coords) - min(x_coords),
                   max(y_coords) - min(y_coords)) * 2.5
        return (int(cx - size), int(cy - size), int(cx + size), int(cy + size))

    elif crop_to == "upper_body":
        keys = ["nose", "left_shoulder", "right_shoulder",
                "left_hip", "right_hip"]
        points = [person_kp[KEYPOINTS[k]][:2] for k in keys]
        valid = [p for p in points if p[0] > 0 and p[1] > 0]
        if len(valid) < 2:
            return tuple(int(v) for v in box)
        return (int(min(p[0] for p in valid)), int(min(p[1] for p in valid)),
                int(max(p[0] for p in valid)), int(max(p[1] for p in valid)))

    elif crop_to == "lower_body":
        keys = ["left_hip", "right_hip", "left_knee", "right_knee",
                "left_ankle", "right_ankle"]
        points = [person_kp[KEYPOINTS[k]][:2] for k in keys]
        valid = [p for p in points if p[0] > 0 and p[1] > 0]
        if len(valid) < 2:
            return tuple(int(v) for v in box)
        return (int(min(p[0] for p in valid)), int(min(p[1] for p in valid)),
                int(max(p[0] for p in valid)), int(max(p[1] for p in valid)))

    elif crop_to == "full_body":
        return tuple(int(v) for v in box)

    return tuple(int(v) for v in box)


def _expand_bbox(x1, y1, x2, y2, img_w, img_h, margin=0.15):
    w, h = x2 - x1, y2 - y1
    mx, my = int(w * margin), int(h * margin)
    return (max(0, x1 - mx), max(0, y1 - my),
            min(img_w, x2 + mx), min(img_h, y2 + my))


def _detect_person(frame, model, intent, region, config):
    result = model(frame)[0]
    if result.keypoints is None or result.boxes is None:
        return False, "No person detected"

    keypoints_all = result.keypoints.data.cpu().numpy()
    boxes = result.boxes.xyxy.cpu().numpy()

    if intent == "basic":
        required_parts, crop_to = ["face"], "face"
    elif intent == "matching":
        required_parts, crop_to = ["face", "hips", "ankles"], "full_body"
    elif intent == "suggestion":
        if region not in REGION_CONFIG:
            return False, f"Invalid region: {region}"
        required_parts = REGION_CONFIG[region]["required_parts"]
        crop_to = REGION_CONFIG[region]["crop_to"]
    else:
        return False, f"Invalid intent: {intent}"

    valid_people = []
    for idx, (kp, box_coords) in enumerate(zip(keypoints_all, boxes)):
        if _validate_required_parts(kp, required_parts, config["kp_conf_thresh"]):
            x1, y1, x2, y2 = box_coords.astype(int)
            valid_people.append({
                'index': idx, 'keypoints': kp,
                'box': (x1, y1, x2, y2),
                'area': (x2 - x1) * (y2 - y1),
                'crop_to': crop_to
            })

    if not valid_people:
        return False, f"Required body parts not visible: {', '.join(required_parts)}"

    selected = max(valid_people, key=lambda p: p['area'])
    return True, selected


print("✅ Detection helpers loaded!")

✅ Detection helpers loaded!


In [ ]:
# ============================================
# CELL 8: GEMINI ANALYSIS — Indian + Western Dress Aware
# ============================================

def analyze_image_with_gemini(img_pil, intent, region=None):
    prompts = {
        "basic": '''Analyze this person carefully. Return ONLY JSON:
{
  "skintone": "Type X (1-6)",
  "gender": "Male/Female"
}''',

        "matching": '''Analyze this person's complete outfit carefully and return ONLY JSON.

DETECTION RULES:

1. Identify gender first.
2. Identify the DRESS TYPE from this list:
   - "shirt_pants" (western separates — shirt/tshirt + pants/jeans/trousers/shorts)
   - "kurta_pajama" (male Indian — kurta/sherwani + pajama/churidar/dhoti)
   - "saree" (female Indian — blouse + saree drape)
   - "churidar" (female Indian — kameez/kurti + churidar/salwar/legging)
   - "lehenga" (female Indian — choli/blouse + lehenga skirt)
   - "anarkali" (female Indian — long flowing anarkali suit)
   - "sharara" (female Indian — short kurti + wide sharara pants)
   - "frock" (one piece short dress)
   - "gown" (one piece long/formal dress)
   - "jumpsuit" (one piece with legs)
   - "mundu_veshti" (South Indian male — mundu/veshti + shirt/jubba)
   - "half_saree" (South Indian female — half saree / langa voni)
   - "other" (anything else)

3. For UPPER and LOWER detection:
   MALES:
   - shirt_pants → upper = shirt/tshirt color+type, lower = pants/jeans color+type
   - kurta_pajama → upper = kurta/sherwani color+type, lower = pajama/churidar/dhoti color
   - mundu_veshti → upper = shirt/jubba color, lower = mundu/veshti color+border

   FEMALES:
   - saree → upper = blouse color+type, lower = saree color+type+border description
   - churidar → upper = kameez/kurti color+type, lower = churidar/salwar color
   - lehenga → upper = choli/blouse color, lower = lehenga skirt color+work
   - anarkali → upper = anarkali color+description, lower = same as upper (one piece)
   - sharara → upper = kurti color, lower = sharara pants color
   - half_saree → upper = blouse color, lower = half saree skirt+dupatta color
   - frock/gown → upper = dress color+type, lower = same as upper (one piece)
   - jumpsuit → upper = jumpsuit color+type, lower = same as upper (one piece)

4. Detect dupatta/shawl/stole if visible → include in description
5. If a region is NOT clearly visible → return "unknown"
6. Detect footwear if visible (type + color)
7. Detect visible accessories (jewelry, watch, belt, etc.)

Return STRICTLY in this JSON format:
{
  "skintone": "Type X (1-6)",
  "gender": "Male/Female",
  "dress_type": "one of the types listed above",
  "upper": "detailed color and type description",
  "lower": "detailed color and type description",
  "footwear": "type and color, or none",
  "accessories": "visible accessories, or none",
  "fabric_guess": "silk/cotton/chiffon/polyester/denim/linen/brocade/unknown",
  "pattern": "solid/printed/striped/checked/embroidered/sequined/plain/unknown"
}''',

        "suggestion_upper": '''Analyze this person's UPPER body clothing carefully.

DETECTION RULES:
1. Identify gender
2. Identify dress type:
   MALES: shirt/tshirt/kurta/sherwani/jubba/jacket/sweater
   FEMALES:
   - If saree → upper = blouse (describe color, sleeve type)
   - If churidar/salwar kameez → upper = kameez/kurti (describe color, length, style)
   - If lehenga → upper = choli/blouse (describe color, work)
   - If anarkali → upper = full anarkali description
   - If frock/gown/jumpsuit → upper = full dress description
   - If western separates → upper = top/shirt/blouse description

3. Note the fabric if identifiable (silk, cotton, chiffon, etc.)
4. Note any embroidery, print, or pattern
5. Note dupatta if visible

Return ONLY JSON:
{
  "skintone": "Type X (1-6)",
  "gender": "Male/Female",
  "dress_type": "type from list above",
  "upper": "detailed color, type, fabric, pattern description",
  "fabric_guess": "silk/cotton/chiffon/polyester/unknown",
  "pattern": "solid/printed/embroidered/unknown"
}''',

        "suggestion_lower": '''Analyze this person's LOWER body clothing carefully.

DETECTION RULES:
1. Identify gender
2. Identify dress type:
   MALES: pants/jeans/trousers/shorts/pajama/dhoti/mundu/veshti/churidar
   FEMALES:
   - If saree → lower = saree drape (describe color, border, fabric)
   - If churidar/salwar → lower = churidar/salwar (describe color)
   - If lehenga → lower = lehenga skirt (describe color, work, flare)
   - If sharara → lower = sharara pants (describe color, width)
   - If half saree → lower = skirt + dupatta description
   - If western → lower = pants/skirt/jeans description

3. Note fabric if identifiable
4. Note pattern/work (zari border, mirror work, embroidery, etc.)

Return ONLY JSON:
{
  "skintone": "Type X (1-6)",
  "gender": "Male/Female",
  "dress_type": "type from list above",
  "lower": "detailed color, type, fabric, pattern description",
  "fabric_guess": "silk/cotton/chiffon/denim/unknown",
  "pattern": "solid/printed/embroidered/bordered/unknown"
}'''
    }

    key = f"suggestion_{region}" if intent == "suggestion" else intent

    try:
        response = gemini_model.generate_content(
            [prompts.get(key, prompts["basic"]), img_pil]
        )
        return response.text
    except Exception as e:
        print(f"❌ Gemini error: {e}")
        return f'{{"error": "{str(e)}"}}'


def parse_gemini_response(response_text):
    try:
        clean = response_text.strip()
        if clean.startswith("```json"):
            clean = clean[7:]
        if clean.startswith("```"):
            clean = clean[3:]
        if clean.endswith("```"):
            clean = clean[:-3]
        return json.loads(clean.strip())
    except (json.JSONDecodeError, ValueError) as e:
        print(f"⚠️ Gemini parse error: {e}")
        return {"raw": response_text}


print("✅ Gemini analysis loaded (Indian + Western dress aware)!")

✅ Gemini analysis loaded (Indian + Western dress aware)!


In [ ]:
# ============================================
# CELL 9: OUTFIT IMAGE GENERATION (Updated for traditional wear)
# ============================================

def _build_outfit_prompt(outfit_data, analysis):
    """Convert outfit data to detailed image generation prompt"""
    gender = analysis.get('gender', 'person').lower()
    skintone = analysis.get('skintone', 'Type 3')

    skintone_map = {
        "Type 1": "very fair skin tone",
        "Type 2": "fair skin tone",
        "Type 3": "light to medium skin tone",
        "Type 4": "olive skin tone",
        "Type 5": "brown skin tone",
        "Type 6": "dark brown skin tone"
    }
    skin_desc = skintone_map.get(skintone, "medium skin tone")

    # Check if it's a single "outfit" key (traditional/one-piece)
    if outfit_data.get('outfit'):
        outfit_desc = outfit_data['outfit']
    else:
        outfit_parts = []
        for key in ['inner', 'upper', 'lower', 'footwear']:
            val = outfit_data.get(key)
            if val and str(val).lower() not in ['none', 'n/a', 'not specified', 'not detected', '']:
                outfit_parts.append(val)

        if not outfit_parts:
            outfit_desc = "stylish casual outfit"
        elif len(outfit_parts) == 1:
            outfit_desc = outfit_parts[0]
        elif len(outfit_parts) == 2:
            outfit_desc = f"{outfit_parts[0]} and {outfit_parts[1]}"
        else:
            outfit_desc = ", ".join(outfit_parts[:-1]) + f", and {outfit_parts[-1]}"

    # Add accessories if present
    accessories = outfit_data.get('accessories', '')
    if accessories and str(accessories).lower() not in ['none', 'n/a', '']:
        outfit_desc += f", accessorized with {accessories}"

    prompt = (
        f"Professional fashion photography of a {gender} fashion model with {skin_desc}, "
        f"wearing {outfit_desc}, standing in confident elegant pose, "
        f"plain light gray studio background, soft professional lighting, "
        f"full body shot, high quality, sharp focus, modern fashion editorial style, "
        f"clean composition, 8k resolution, photorealistic, detailed fabric textures visible"
    )
    return prompt


def generate_outfit_image(recommendation, analysis):
    """Generate outfit visualization using fal.ai"""

    # Merge user's existing clothing with recommendations
    full_outfit = {}

    # If recommendation has 'outfit' key (traditional/one-piece)
    if recommendation.get('outfit'):
        full_outfit['outfit'] = recommendation['outfit']
    else:
        # Start with what user is wearing
        for key in ['upper', 'inner', 'lower', 'footwear']:
            existing = analysis.get(key)
            if existing and str(existing).lower() not in ['none', 'n/a', 'not specified', 'not detected', '']:
                full_outfit[key] = existing

        # Override with recommendations
        for key in ['upper', 'inner', 'lower', 'footwear']:
            rec_val = recommendation.get(key)
            if rec_val and str(rec_val).lower() not in ['none', 'n/a', '']:
                full_outfit[key] = rec_val

    # Add accessories
    if recommendation.get('accessories'):
        full_outfit['accessories'] = recommendation['accessories']

    prompt = _build_outfit_prompt(full_outfit, analysis)
    print(f"🎨 Generating image with prompt: {prompt[:120]}...")

    try:
        result = fal_client.run(
            "fal-ai/flux/schnell",
            arguments={
                "prompt": prompt,
                "image_size": "square_hd",
                "num_inference_steps": 4,
                "num_images": 1,
                "enable_safety_checker": True
            }
        )

        image_url = result['images'][0]['url']
        print(f"✅ Image generated: {image_url}")

        img_response = requests.get(image_url, timeout=30)
        img_response.raise_for_status()
        img_b64 = b64encode(img_response.content).decode('utf-8')

        return {
            "success": True,
            "image_b64": img_b64,
            "prompt": prompt
        }

    except Exception as e:
        error_msg = str(e)
        print(f"❌ Image generation failed: {error_msg}")
        return {
            "success": False,
            "error": error_msg,
            "prompt": prompt
        }


print("✅ Outfit image generation loaded!")

✅ Outfit image generation loaded!


In [ ]:
# ============================================
# CELL 10: FASHION RECOMMENDATION — Diverse + Indian Traditional + Smart Rating
# ============================================

def _extract_json_from_text(text):
    """Robustly extract JSON object from LLM text"""
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    text = text.strip()
    try:
        return json.loads(text)
    except (json.JSONDecodeError, ValueError):
        pass
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except (json.JSONDecodeError, ValueError):
            pass
    return None


# ============================================
# OCCASION → DRESS STYLE MAPPING
# ============================================

OCCASION_CATEGORIES = {
    "indian_wedding_ceremony": {
        "tier": "grand_traditional",
        "male": ["sherwani with churidar", "bandhgala suit with dhoti", "silk kurta with pajama and dupatta"],
        "female": ["heavy silk saree with temple jewelry", "bridal lehenga choli", "kanjeevaram silk saree", "banarasi saree with gold jewelry"],
        "colors_light": ["deep red", "maroon", "royal blue", "emerald green", "gold"],
        "colors_medium": ["wine red", "teal", "magenta", "burnt orange", "deep purple"],
        "colors_dark": ["bright red", "hot pink", "gold", "turquoise", "royal blue", "coral"]
    },
    "sangeet": {
        "tier": "festive_glam",
        "male": ["embroidered kurta with jodhpuri pants", "indo-western jacket with kurta", "nehru jacket over silk shirt with trousers"],
        "female": ["sequin lehenga", "sharara suit with mirror work", "anarkali suit with dupatta", "designer saree with statement jewelry"],
        "colors_light": ["coral", "turquoise", "lavender", "mint green", "peach"],
        "colors_medium": ["mustard", "teal", "magenta", "rust", "emerald"],
        "colors_dark": ["bright pink", "yellow", "electric blue", "coral", "gold"]
    },
    "mehndi_haldi": {
        "tier": "festive_casual",
        "male": ["printed kurta with white pajama", "yellow kurta with dhoti", "floral kurta with jeans"],
        "female": ["yellow sharara suit", "floral anarkali", "cotton lehenga with gota patti work", "printed saree with flower jewelry"],
        "colors_light": ["yellow", "marigold", "light green", "peach", "coral"],
        "colors_medium": ["turmeric yellow", "orange", "green", "lime", "saffron"],
        "colors_dark": ["bright yellow", "orange", "lime green", "hot pink", "gold"]
    },
    "reception": {
        "tier": "formal_elegant",
        "male": ["three-piece suit", "tuxedo with bow tie", "bandhgala suit", "indo-western sherwani"],
        "female": ["designer gown", "cocktail saree with modern draping", "lehenga with cape", "embroidered floor-length anarkali"],
        "colors_light": ["navy", "charcoal", "deep wine", "forest green", "champagne"],
        "colors_medium": ["black", "midnight blue", "deep maroon", "dark teal", "plum"],
        "colors_dark": ["black", "ivory", "gold", "silver", "royal blue", "deep red"]
    },
    "diwali": {
        "tier": "festive_traditional",
        "male": ["silk kurta pajama set", "brocade kurta with churidar", "nehru jacket with kurta and white pajama"],
        "female": ["silk saree with gold border", "anarkali suit with dupatta", "festive lehenga choli", "chanderi silk saree"],
        "colors_light": ["deep red", "gold", "maroon", "emerald", "royal blue"],
        "colors_medium": ["wine", "mustard gold", "teal", "burgundy", "copper"],
        "colors_dark": ["bright red", "gold", "magenta", "orange", "turquoise"]
    },
    "onam": {
        "tier": "traditional_regional",
        "male": ["cream mundu with gold kasavu border and white shirt", "double mundu with jubba", "off-white dhoti with silk kurta"],
        "female": ["cream and gold Kerala kasavu saree (settu mundu)", "off-white Kerala saree with gold tissue border", "cream tissue half saree with gold jewelry"],
        "colors_light": ["cream with gold", "off-white with gold", "ivory with gold border"],
        "colors_medium": ["cream with gold", "off-white with gold zari", "ivory with copper border"],
        "colors_dark": ["cream with gold", "off-white with gold", "white with gold and green border"]
    },
    "pongal": {
        "tier": "traditional_regional",
        "male": ["white veshti with silk shirt", "cream dhoti kurta set", "traditional white mundu with angavastram"],
        "female": ["bright silk saree (pattu pudavai)", "kanjeevaram silk saree with traditional jewelry", "cotton saree with temple border"],
        "colors_light": ["deep red", "mustard", "green", "orange", "gold"],
        "colors_medium": ["maroon with gold", "forest green", "saffron", "rust", "magenta"],
        "colors_dark": ["bright yellow", "red with gold", "orange", "hot pink", "emerald"]
    },
    "eid": {
        "tier": "festive_traditional",
        "male": ["embroidered pathani suit", "sherwani with pajama", "kurta with salwar and waistcoat", "lucknowi chikan kurta with churidar"],
        "female": ["anarkali suit with embroidered dupatta", "sharara set with chikankari work", "gharara suit with zari dupatta", "embroidered salwar kameez"],
        "colors_light": ["pastel green", "ivory", "powder blue", "soft pink", "mint"],
        "colors_medium": ["emerald", "teal", "peach", "dusty rose", "sage green"],
        "colors_dark": ["white", "bright green", "turquoise", "coral", "royal blue"]
    },
    "christmas": {
        "tier": "smart_festive",
        "male": ["red sweater with dark jeans", "blazer with christmas themed tie", "smart casual shirt with chinos"],
        "female": ["red dress", "green blouse with white skirt", "sequin top with pants", "velvet dress"],
        "colors_light": ["red", "forest green", "gold", "burgundy", "ivory"],
        "colors_medium": ["deep red", "emerald", "gold", "maroon", "hunter green"],
        "colors_dark": ["bright red", "white", "gold", "silver", "emerald"]
    },
    "vishu": {
        "tier": "traditional_regional",
        "male": ["kasavu mundu with white shirt", "cream dhoti with gold border and jubba", "off-white kurta pajama"],
        "female": ["kasavu saree (cream with gold border)", "Kerala style half saree", "off-white saree with gold tissue"],
        "colors_light": ["cream with gold", "off-white", "ivory with gold"],
        "colors_medium": ["cream with gold", "off-white with gold zari", "ivory"],
        "colors_dark": ["cream with gold", "off-white with gold", "white with gold"]
    },
    "navratri_garba": {
        "tier": "festive_dance",
        "male": ["embroidered kediyu with dhoti", "colorful kurta with jacket and dhoti", "mirror work kurta with white pajama"],
        "female": ["chaniya choli with mirror work", "colorful ghaghra choli with dupatta", "bandhani print lehenga", "embroidered garba dress"],
        "colors_light": ["bright pink", "orange", "turquoise", "red", "multicolor"],
        "colors_medium": ["magenta", "orange", "red", "electric blue", "multicolor"],
        "colors_dark": ["bright orange", "hot pink", "yellow", "red", "multicolor"]
    },
    "holi": {
        "tier": "ultra_casual_festive",
        "male": ["old white kurta pajama", "white t-shirt with white pajama", "simple white cotton kurta"],
        "female": ["white cotton kurta with leggings", "old white salwar kameez", "simple white kurti with white pajama"],
        "colors_light": ["white", "off-white"],
        "colors_medium": ["white", "off-white"],
        "colors_dark": ["white", "off-white"]
    },
    "raksha_bandhan": {
        "tier": "semi_traditional",
        "male": ["kurta with jeans", "printed kurta with white pajama", "nehru jacket with shirt"],
        "female": ["ethnic kurti with palazzo", "light anarkali suit", "printed saree", "sharara suit"],
        "colors_light": ["coral", "pink", "lavender", "mint", "peach"],
        "colors_medium": ["rose", "teal", "mustard", "dusty pink", "sage"],
        "colors_dark": ["bright pink", "coral", "turquoise", "yellow", "magenta"]
    },
    "ganesh_chaturthi": {
        "tier": "festive_traditional",
        "male": ["dhoti kurta set", "silk kurta with churidar", "traditional kurta pajama"],
        "female": ["nauvari saree (Maharashtrian style)", "silk saree", "traditional salwar kameez", "lehenga with ethnic jewelry"],
        "colors_light": ["red", "orange", "yellow", "green", "gold"],
        "colors_medium": ["saffron", "maroon", "mustard", "orange red", "green"],
        "colors_dark": ["bright orange", "red", "yellow", "gold", "green"]
    },
    "black_tie": {
        "tier": "ultra_formal",
        "male": ["black tuxedo with bow tie and patent shoes", "midnight blue tuxedo with white dress shirt"],
        "female": ["floor-length evening gown", "elegant cocktail dress with heels", "designer ball gown"],
        "colors_light": ["black", "midnight blue", "deep charcoal"],
        "colors_medium": ["black", "midnight blue", "deep navy"],
        "colors_dark": ["black", "midnight blue", "ivory", "gold accent"]
    },
    "job_interview": {
        "tier": "business_formal",
        "male": ["navy suit with light blue shirt and tie", "charcoal suit with white shirt", "dark grey suit with subtle tie"],
        "female": ["tailored blazer with formal trousers", "pencil skirt with blouse and blazer", "formal kurta with trousers"],
        "colors_light": ["navy", "charcoal", "dark grey", "deep blue"],
        "colors_medium": ["navy", "dark grey", "charcoal", "deep brown"],
        "colors_dark": ["navy", "charcoal", "black", "deep grey"]
    },
    "office_work": {
        "tier": "business_casual",
        "male": ["chinos with button-down shirt", "smart trousers with polo", "kurta with formal trousers"],
        "female": ["kurti with straight pants", "blouse with formal trousers", "cotton saree", "smart kurta with palazzo"],
        "colors_light": ["light blue", "white", "pastel pink", "sage", "lavender"],
        "colors_medium": ["olive", "maroon", "teal", "mustard", "rust"],
        "colors_dark": ["bright blue", "white", "coral", "turquoise", "emerald"]
    },
    "date_night": {
        "tier": "smart_casual",
        "male": ["dark jeans with fitted shirt and blazer", "chinos with knit sweater", "smart kurta with jeans"],
        "female": ["wrap dress", "fitted jeans with elegant top", "kurti with statement earrings and palazzo", "midi dress with heels"],
        "colors_light": ["burgundy", "navy", "forest green", "deep plum", "charcoal"],
        "colors_medium": ["maroon", "deep teal", "wine", "olive", "rust"],
        "colors_dark": ["bright red", "royal blue", "white", "emerald", "coral"]
    },
    "party_club": {
        "tier": "party_glam",
        "male": ["black shirt with dark jeans", "leather jacket with fitted tee", "printed shirt with dark trousers"],
        "female": ["bodycon dress", "sequin top with pants", "party saree with modern draping", "jumpsuit with heels"],
        "colors_light": ["black", "dark red", "navy", "metallic silver"],
        "colors_medium": ["black", "deep red", "emerald", "metallics"],
        "colors_dark": ["black", "gold", "silver", "bright red", "white"]
    },
    "casual_everyday": {
        "tier": "casual",
        "male": ["t-shirt with jeans and sneakers", "polo with shorts", "casual kurta with jeans"],
        "female": ["kurti with jeans", "casual dress with flats", "t-shirt with jeans", "cotton kurta with leggings"],
        "colors_light": ["pastel blue", "sage", "soft pink", "cream", "lavender"],
        "colors_medium": ["olive", "rust", "teal", "mustard", "terracotta"],
        "colors_dark": ["bright blue", "yellow", "coral", "white", "turquoise"]
    },
    "beach_vacation": {
        "tier": "ultra_casual",
        "male": ["linen shirt with shorts and flip flops", "printed hawaiian shirt with swim shorts", "cotton kurta with shorts"],
        "female": ["sundress with sandals", "sarong with bikini top", "cotton maxi dress", "linen kurti with shorts"],
        "colors_light": ["coral", "turquoise", "yellow", "white", "mint"],
        "colors_medium": ["bright blue", "orange", "teal", "coral", "lime"],
        "colors_dark": ["bright yellow", "white", "coral", "turquoise", "orange"]
    },
    "temple_pooja": {
        "tier": "modest_traditional",
        "male": ["dhoti with simple kurta", "white veshti with white shirt", "simple kurta pajama"],
        "female": ["cotton saree with minimal jewelry", "simple salwar kameez", "modest kurti with churidar"],
        "colors_light": ["yellow", "cream", "light pink", "white", "pastel orange"],
        "colors_medium": ["saffron", "cream", "light yellow", "orange", "red"],
        "colors_dark": ["yellow", "cream", "saffron", "orange", "white"]
    },
    "college_campus": {
        "tier": "trendy_casual",
        "male": ["graphic tee with joggers and sneakers", "oversized shirt with cargo pants", "kurta with ripped jeans"],
        "female": ["crop top with high-waist jeans", "kurti with jeans and kolhapuri chappals", "oversized tee with skirt", "ethnic jacket with casual outfit"],
        "colors_light": ["pastel purple", "baby blue", "mint", "peach", "cream"],
        "colors_medium": ["olive", "burnt orange", "teal", "mustard", "denim blue"],
        "colors_dark": ["bright yellow", "electric blue", "coral", "white", "neon green"]
    },
    "funeral": {
        "tier": "somber_formal",
        "male": [
            "plain white kurta with white pajama (Hindu tradition)",
            "plain white dhoti with white shirt",
            "simple black suit with white shirt",
            "plain white sherwani (close family)"
        ],
        "female": [
            "plain white cotton saree without border",
            "plain white salwar kameez with dupatta",
            "simple white or off-white kurta with white dupatta",
            "plain white or muted cotton saree (no embellishments)"
        ],
        "colors_light": ["white", "off-white", "pale grey", "ivory"],
        "colors_medium": ["white", "light grey", "muted beige", "pale cream"],
        "colors_dark": ["white", "off-white", "soft grey", "ivory"],
        "notes": [
            "avoid bright colors and heavy embellishments",
            "white is traditional for Hindu funerals",
            "black is acceptable in urban and Christian contexts",
            "keep jewelry minimal or avoid entirely",
            "avoid prints and patterns"
        ]
    },

    "swimming": {
        "tier": "athletic_casual",
        "male": [
            "swim trunks / board shorts",
            "athletic jammers (competitive)",
            "rash guard with swim shorts",
            "wetsuit (if required)"
        ],
        "female": [
            "one-piece swimsuit",
            "bikini / two-piece swimsuit",
            "tankini with swim shorts",
            "rash guard with leggings (modest option)",
            "burkini (full-coverage option)"
        ],
        "colors_light": ["navy", "black", "dark teal", "forest green"],
        "colors_medium": ["cobalt blue", "teal", "red", "black", "charcoal"],
        "colors_dark": ["bright blue", "neon green", "hot pink", "yellow", "orange", "coral"],
        "notes": [
            "choose chlorine-resistant fabric for pool use",
            "UV protection rash guards recommended for outdoor swimming",
            "avoid cotton as it absorbs water and becomes heavy",
            "water shoes optional for rocky beaches or pool decks"
        ]
    },

    "graduation_day": {
        "tier": "formal_celebratory",
        "male": [
            "academic gown over formal shirt and trousers",
            "navy or charcoal suit with tie under graduation gown",
            "white formal shirt with dress trousers (if no gown required)",
            "bandhgala suit (Indian formal alternative)"
        ],
        "female": [
            "academic gown over formal dress or saree",
            "midi dress or formal skirt suit under graduation gown",
            "formal saree or salwar kameez (Indian tradition)",
            "blazer with formal trousers or skirt",
            "shirt dress or wrap dress in solid colors"
        ],
        "colors_light": ["navy", "charcoal", "black", "deep maroon", "forest green"],
        "colors_medium": ["black", "midnight blue", "dark grey", "deep burgundy", "dark teal"],
        "colors_dark": ["black", "ivory", "royal blue", "deep red", "gold", "silver"],
        "notes": [
            "graduation gown and cap are typically provided by the institution",
            "wear comfortable shoes as ceremonies can be long",
            "avoid very bright or distracting colors under the gown",
            "coordinate with the institution's dress code if specified",
            "Indian graduates often wear traditional attire under the gown"
        ]
    }
}


OCCASION_KEYWORD_MAP = {
    "indian wedding": "indian_wedding_ceremony", "hindu wedding": "indian_wedding_ceremony",
    "wedding ceremony": "indian_wedding_ceremony", "shaadi": "indian_wedding_ceremony",
    "vivah": "indian_wedding_ceremony", "kalyanam": "indian_wedding_ceremony",
    "sangeet": "sangeet", "sangeet night": "sangeet",
    "mehndi": "mehndi_haldi", "haldi": "mehndi_haldi", "mehendi": "mehndi_haldi",
    "reception": "reception", "wedding reception": "reception",
    "engagement": "sangeet", "sagai": "sangeet",
    "diwali": "diwali", "deepavali": "diwali",
    "onam": "onam", "thiruvonam": "onam",
    "pongal": "pongal", "thai pongal": "pongal", "sankranti": "pongal", "makar sankranti": "pongal",
    "eid": "eid", "eid ul fitr": "eid", "eid ul adha": "eid", "ramadan": "eid",
    "christmas": "christmas", "xmas": "christmas",
    "vishu": "vishu", "ugadi": "vishu", "gudi padwa": "vishu",
    "navratri": "navratri_garba", "garba": "navratri_garba", "dandiya": "navratri_garba", "durga puja": "navratri_garba",
    "holi": "holi",
    "raksha bandhan": "raksha_bandhan", "rakhi": "raksha_bandhan", "bhai dooj": "raksha_bandhan",
    "ganesh chaturthi": "ganesh_chaturthi", "ganpati": "ganesh_chaturthi", "vinayaka chaturthi": "ganesh_chaturthi",
    "lohri": "diwali", "baisakhi": "navratri_garba", "bihu": "navratri_garba", "karva chauth": "diwali",
    "black tie": "black_tie", "gala": "black_tie", "ball": "black_tie", "opera": "black_tie",
    "job interview": "job_interview", "interview": "job_interview", "board meeting": "job_interview",
    "court": "job_interview", "funeral": "job_interview",
    "office": "office_work", "work": "office_work", "meeting": "office_work",
    "conference": "office_work", "presentation": "office_work",
    "date": "date_night", "date night": "date_night", "dinner": "date_night",
    "restaurant": "date_night", "anniversary": "date_night",
    "party": "party_club", "club": "party_club", "nightclub": "party_club",
    "cocktail": "party_club", "new year": "party_club", "new years eve": "party_club",
    "birthday party": "party_club", "house party": "party_club",
    "casual": "casual_everyday", "everyday": "casual_everyday", "weekend": "casual_everyday",
    "shopping": "casual_everyday", "movies": "casual_everyday", "friends": "casual_everyday",
    "hangout": "casual_everyday", "coffee": "casual_everyday", "brunch": "casual_everyday",
    "beach": "beach_vacation", "vacation": "beach_vacation", "holiday": "beach_vacation",
    "picnic": "beach_vacation", "bbq": "beach_vacation",
    "temple": "temple_pooja", "pooja": "temple_pooja", "puja": "temple_pooja",
    "church": "temple_pooja", "mosque": "temple_pooja", "gurudwara": "temple_pooja", "prayer": "temple_pooja",
    "college": "college_campus", "campus": "college_campus", "university": "college_campus", "school": "college_campus",
    "traditional": "diwali", "ethnic": "diwali", "festival": "diwali",
    "function": "sangeet", "ceremony": "indian_wedding_ceremony", "celebration": "sangeet",
    "western wedding": "reception", "formal wedding": "black_tie", "church wedding": "reception",
}


def _get_occasion_category(occasion_str):
    occ = occasion_str.lower().strip()
    if occ in OCCASION_KEYWORD_MAP:
        return OCCASION_KEYWORD_MAP[occ]
    sorted_keys = sorted(OCCASION_KEYWORD_MAP.keys(), key=len, reverse=True)
    for keyword in sorted_keys:
        if keyword in occ:
            return OCCASION_KEYWORD_MAP[keyword]
    return "casual_everyday"


def _get_skin_tone_tier(skintone):
    skintone = skintone.lower()
    if "1" in skintone or "2" in skintone:
        return "light"
    elif "3" in skintone or "4" in skintone:
        return "medium"
    elif "5" in skintone or "6" in skintone:
        return "dark"
    return "medium"


# ============================================
# INDIAN DRESS RATING KNOWLEDGE BASE
# ============================================

DRESS_TYPE_RATING_RULES = {
    "saree": {
        "good_combos": [
            "contrast blouse with saree body color",
            "matching blouse with saree border color",
            "gold/silver blouse with silk saree",
            "same color family blouse with printed saree",
        ],
        "bad_combos": [
            "neon blouse with traditional silk saree",
            "casual t-shirt material with silk saree",
            "completely clashing uncoordinated colors",
        ],
        "fabric_expectations": ["silk", "chiffon", "georgette", "cotton", "organza", "net", "crepe"],
        "formality": "semi-formal to formal",
        "style_tips": "Blouse should complement saree — either match border color or provide elegant contrast. Silk sarees pair best with silk or brocade blouses."
    },
    "churidar": {
        "good_combos": [
            "matching or coordinating dupatta",
            "contrast bottom with printed top",
            "same fabric top and bottom",
            "embroidered top with plain bottom",
        ],
        "bad_combos": [
            "heavy embroidered top with heavy embroidered bottom",
            "completely mismatched fabric weights",
            "clashing prints on both top and bottom",
        ],
        "fabric_expectations": ["cotton", "silk", "georgette", "chiffon", "crepe"],
        "formality": "casual to semi-formal",
        "style_tips": "If top is heavily worked, keep bottom plain. Dupatta should tie the look together."
    },
    "lehenga": {
        "good_combos": [
            "matching choli with lehenga work",
            "contrast choli that picks up lehenga border color",
            "gold/silver choli with richly colored lehenga",
            "same fabric choli and lehenga",
        ],
        "bad_combos": [
            "casual cotton choli with heavy lehenga",
            "completely unrelated color choli",
            "plain choli with plain lehenga (looks incomplete)",
        ],
        "fabric_expectations": ["silk", "brocade", "velvet", "net", "organza", "georgette"],
        "formality": "semi-formal to grand formal",
        "style_tips": "Choli should complement lehenga — pick a color from lehenga's embroidery or border. Heavy lehenga pairs well with fitted choli."
    },
    "kurta_pajama": {
        "good_combos": [
            "white/cream pajama with colored kurta",
            "matching tone pajama with kurta",
            "contrast nehru jacket over plain kurta",
            "silk kurta with silk pajama for formal",
        ],
        "bad_combos": [
            "heavy printed pajama with heavy printed kurta",
            "jeans with very formal silk kurta",
            "sports shoes with traditional kurta",
        ],
        "fabric_expectations": ["cotton", "silk", "linen", "khadi", "brocade"],
        "formality": "casual to formal depending on fabric",
        "style_tips": "Keep bottom simpler than top. White/cream pajama is safest. Dupatta adds formality for women's kurta sets."
    },
    "mundu_veshti": {
        "good_combos": [
            "white shirt with white mundu and gold border",
            "cream jubba with kasavu mundu",
            "simple shirt with double mundu",
        ],
        "bad_combos": [
            "printed casual tee with formal mundu",
            "dark colored shirt with kasavu mundu",
            "jeans jacket over mundu",
        ],
        "fabric_expectations": ["cotton", "silk", "handloom"],
        "formality": "traditional formal",
        "style_tips": "White/cream top works best. Gold kasavu border adds formality. Keep it simple and elegant."
    },
    "anarkali": {
        "good_combos": [
            "flowing anarkali with matching dupatta",
            "contrast dupatta that picks accent color",
            "matching churidar underneath",
        ],
        "bad_combos": [
            "heavy chunky jewelry with already heavy embroidered anarkali",
            "casual sneakers with formal anarkali",
        ],
        "fabric_expectations": ["georgette", "silk", "net", "chiffon", "crepe"],
        "formality": "semi-formal to formal",
        "style_tips": "Anarkali is a complete look — accessories should enhance not overwhelm."
    },
    "half_saree": {
        "good_combos": [
            "matching blouse with skirt color",
            "contrast dupatta as highlight",
            "coordinating jewelry with outfit colors",
        ],
        "bad_combos": [
            "mismatched blouse that clashes with skirt",
            "western accessories with traditional half saree",
        ],
        "fabric_expectations": ["silk", "cotton", "organza", "net"],
        "formality": "semi-formal",
        "style_tips": "Three pieces (blouse, skirt, dupatta) should have color harmony — at least two should coordinate."
    },
    "shirt_pants": {
        "good_combos": [
            "neutral pants with colored/patterned shirt",
            "dark jeans with light shirt",
            "matching belt and shoes",
            "blazer that coordinates with shirt color",
        ],
        "bad_combos": [
            "two bold clashing patterns",
            "brown belt with black shoes",
            "very formal shirt with casual shorts",
        ],
        "fabric_expectations": ["cotton", "polyester", "denim", "linen"],
        "formality": "casual to business casual",
        "style_tips": "One statement piece — if shirt is bold, pants should be neutral. Shoes and belt should coordinate."
    },
    "frock": {
        "good_combos": ["solid color with complementary accessories", "printed dress with neutral shoes"],
        "bad_combos": ["overly busy pattern with busy accessories", "very casual shoes with formal dress"],
        "fabric_expectations": ["cotton", "chiffon", "polyester", "silk"],
        "formality": "casual to semi-formal",
        "style_tips": "One-piece dresses are self-coordinated — focus on accessories and shoes."
    },
    "gown": {
        "good_combos": ["elegant heels with flowing gown", "minimal jewelry with heavy gown", "statement jewelry with simple gown"],
        "bad_combos": ["casual sandals with formal gown", "heavy chunky jewelry with heavy embellished gown"],
        "fabric_expectations": ["silk", "satin", "chiffon", "velvet", "organza"],
        "formality": "formal to ultra-formal",
        "style_tips": "Less is more with gowns — let the gown be the star. Heels are essential."
    },
    "sharara": {
        "good_combos": ["short kurti with wide sharara", "matching dupatta", "contrast kurti with solid sharara"],
        "bad_combos": ["long kurti that hides sharara silhouette", "very heavy top with heavy bottom"],
        "fabric_expectations": ["georgette", "silk", "chiffon", "net", "cotton"],
        "formality": "semi-formal to formal",
        "style_tips": "Sharara's wide legs are the highlight — kurti should be short to show them. Dupatta ties the look."
    },
    "jumpsuit": {
        "good_combos": ["belt to define waist", "statement earrings", "heels for elongation"],
        "bad_combos": ["too many accessories", "very casual shoes with dressy jumpsuit"],
        "fabric_expectations": ["cotton", "polyester", "crepe", "denim"],
        "formality": "casual to semi-formal",
        "style_tips": "Jumpsuits are complete outfits — add a belt and good shoes."
    }
}


def get_fashion_recommendation(user_input, max_retries=2):
    intent = user_input.get('intent', 'basic').lower()

    system_prompt = """You are an expert Fashion Stylist AI with deep knowledge of BOTH Western AND Indian/South Asian fashion.

CRITICAL RULES:
1. Return ONLY valid JSON — no markdown, no explanation, no extra text
2. Match outfit formality EXACTLY to the occasion
3. Match colors to the user's skin tone
4. Be SPECIFIC — mention fabric, color shade, and style details
5. Include traditional Indian wear when appropriate
6. For Indian festivals/occasions, ALWAYS recommend traditional Indian attire
7. For Western occasions, recommend Western attire
8. For ambiguous occasions, blend Indo-Western styles

SKIN TONE COLOR MATCHING:
TYPE 1-2 (Fair/Light): Deep jewel tones — Navy, Burgundy, Forest Green, Deep Purple. Pastels — Lavender, Baby Blue, Soft Pink, Mint. AVOID washed out yellows, pale beige.
TYPE 3-4 (Medium/Olive): Earth tones — Olive, Rust, Maroon, Teal, Mustard, Terracotta, Burnt Orange, Deep Coral, Copper.
TYPE 5-6 (Dark/Deep): Bold bright — Royal Blue, Emerald, Coral, White, Magenta, Bright Yellow, Turquoise, Gold. AVOID very dark colors with no contrast.

INDIAN DRESS KNOWLEDGE:
MALE: Kurta-pajama, Sherwani, Nehru jacket, Dhoti, Mundu/Veshti, Bandhgala, Pathani suit, Achkan, Kediyu, Jodhpuri suit
FEMALE: Saree (Kanjeevaram, Banarasi, Chanderi, Chiffon, Cotton, Kasavu, Pattu), Lehenga choli, Salwar kameez, Anarkali, Sharara, Gharara, Kurti, Palazzo suit, Half saree, Chaniya choli

OUTPUT FORMAT — Return ONLY valid JSON:
For separates: {"upper":"...","inner":"...(optional)","lower":"...","footwear":"...","accessories":"...(optional)","reason":"..."}
For traditional one-piece (saree/sherwani/gown): {"outfit":"full description","footwear":"...","accessories":"...","reason":"..."}
For rating: {"rate": <0-10>, "reason": "<specific feedback>", "tips": "<improvement suggestions>"}"""

    if intent == "basic":
        occasion = user_input.get('occasion', 'none').lower().strip()
        skintone = user_input.get('skintone', 'Type 3')
        gender = user_input.get('gender', 'Male').lower()
        skin_tier = _get_skin_tone_tier(skintone)

        if occasion in ['none', '']:
            category_key = "casual_everyday"
        else:
            category_key = _get_occasion_category(occasion)

        category = OCCASION_CATEGORIES.get(category_key, OCCASION_CATEGORIES["casual_everyday"])
        gender_key = "male" if "male" in gender or "man" in gender or "boy" in gender else "female"
        outfit_examples = category.get(gender_key, [])
        color_key = f"colors_{skin_tier}"
        suggested_colors = category.get(color_key, category.get("colors_medium", []))

        user_prompt = f"""Recommend ONE complete outfit:

PERSON: {gender_key}, Skin tone: {skintone}
OCCASION: {occasion if occasion not in ['none', ''] else 'everyday casual'}
OCCASION TYPE: {category.get('tier', 'casual')}

SUITABLE STYLES (pick one and customize):
{chr(10).join(f'  - {ex}' for ex in outfit_examples)}

BEST COLORS for their skin tone:
{', '.join(suggested_colors)}

Be specific about fabric, color shade, and style details.
For traditional outfits (saree, sherwani, lehenga): {{"outfit":"...","footwear":"...","accessories":"...","reason":"..."}}
For separates (shirt+pants, kurta+pajama): {{"upper":"...","lower":"...","footwear":"...","reason":"..."}}
Return ONLY JSON:"""

    elif intent == "matching":
        dress_type = user_input.get('dress_type', 'unknown')
        upper = user_input.get('upper', 'Not specified')
        lower = user_input.get('lower', 'Not specified')
        footwear = user_input.get('footwear', 'none')
        accessories = user_input.get('accessories', 'none')
        fabric = user_input.get('fabric_guess', 'unknown')
        pattern = user_input.get('pattern', 'unknown')
        skintone = user_input.get('skintone', 'Type 3')
        gender = user_input.get('gender', 'Male')
        occasion = user_input.get('occasion', 'none')

        # Get dress-specific rating rules
        dress_rules = DRESS_TYPE_RATING_RULES.get(dress_type, DRESS_TYPE_RATING_RULES.get("shirt_pants", {}))
        good_combos = dress_rules.get("good_combos", [])
        bad_combos = dress_rules.get("bad_combos", [])
        style_tips = dress_rules.get("style_tips", "")
        expected_formality = dress_rules.get("formality", "casual")

        user_prompt = f"""Rate this outfit combination (0-10).

PERSON: {gender}, Skin tone: {skintone}
OCCASION: {occasion}

DETECTED OUTFIT:
- Dress Type: {dress_type}
- Upper: {upper}
- Lower: {lower}
- Footwear: {footwear}
- Accessories: {accessories}
- Fabric: {fabric}
- Pattern: {pattern}

RATING CRITERIA FOR "{dress_type}" TYPE OUTFITS:

GOOD combinations (higher score):
{chr(10).join(f'  ✓ {c}' for c in good_combos)}

BAD combinations (lower score):
{chr(10).join(f'  ✗ {c}' for c in bad_combos)}

STYLE GUIDE: {style_tips}
EXPECTED FORMALITY: {expected_formality}

SCORING GUIDE:
9-10: Perfect coordination, colors complement skin tone, occasion-appropriate, great style
7-8: Good overall, minor improvements possible
5-6: Decent but noticeable issues (color clash, formality mismatch, etc.)
3-4: Several problems (wrong occasion dress, bad color combo, mismatched pieces)
1-2: Major issues (completely wrong, very clashing, inappropriate)

Be SPECIFIC in feedback. If it's Indian traditional wear, judge by traditional standards.
If western wear, judge by western standards. Don't mix criteria.

Return ONLY: {{"rate": <0-10>, "reason": "<what works and what doesn't>", "tips": "<1-2 specific improvements>"}}"""

    elif intent == "suggestion":
        upper = user_input.get('upper', '').strip()
        lower = user_input.get('lower', '').strip()
        dress_type = user_input.get('dress_type', 'unknown')
        skintone = user_input.get('skintone', 'Type 3')
        gender = user_input.get('gender', 'Male')
        occasion = user_input.get('occasion', 'none')
        skin_tier = _get_skin_tone_tier(skintone)
        fabric = user_input.get('fabric_guess', 'unknown')
        pattern = user_input.get('pattern', 'unknown')

        # Get dress-specific rules for good combos
        dress_rules = DRESS_TYPE_RATING_RULES.get(dress_type, {})
        style_tips = dress_rules.get("style_tips", "Match the style and formality of existing piece.")

        if upper and upper.lower() not in ['none', 'not detected', 'unknown']:
            user_prompt = f"""This person is wearing: {upper} (upper body)
Dress type detected: {dress_type}
Fabric: {fabric}, Pattern: {pattern}
They need: matching lower body clothing + footwear

Person: {gender}, Skin tone: {skintone}
Occasion: {occasion}

STYLE RULE: {style_tips}

IMPORTANT — match the style:
- Saree blouse → suggest saree drape details
- Kurta/kurti → suggest churidar/palazzo/salwar
- Choli → suggest lehenga details
- Western shirt/top → suggest pants/skirt/jeans
- Jubba → suggest mundu/dhoti

Use colors that work with {skintone} skin tone.
Return ONLY: {{"lower":"...","footwear":"...","accessories":"...(optional)","reason":"..."}}"""

        elif lower and lower.lower() not in ['none', 'not detected', 'unknown']:
            user_prompt = f"""This person is wearing: {lower} (lower body)
Dress type detected: {dress_type}
Fabric: {fabric}, Pattern: {pattern}
They need: matching upper body clothing + footwear

Person: {gender}, Skin tone: {skintone}
Occasion: {occasion}

STYLE RULE: {style_tips}

IMPORTANT — match the style:
- Saree drape → suggest matching blouse
- Churidar/salwar → suggest kameez/kurti
- Lehenga skirt → suggest choli/blouse
- Mundu/veshti → suggest shirt/jubba
- Western pants/jeans → suggest shirt/top/blouse

Use colors that work with {skintone} skin tone.
Return ONLY: {{"upper":"...","footwear":"...","accessories":"...(optional)","reason":"..."}}"""
        else:
            return json.dumps({"error": "Could not detect what you're wearing. Please try again."})
    else:
        return json.dumps({"error": f"Unknown intent: {intent}"})

    # Call LLM with retry
    for attempt in range(max_retries + 1):
        try:
            response = chatbot_client.chat.completions.create(
                model="deepseek/deepseek-chat",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ]
            )
            result_text = response.choices[0].message.content or ""
            parsed = _extract_json_from_text(result_text)

            if parsed is not None:
                if intent == "matching" and "rate" not in parsed:
                    if attempt < max_retries:
                        print(f"⚠️ Retry {attempt + 1}: Missing 'rate' in response")
                        continue
                return json.dumps(parsed)

            if attempt < max_retries:
                print(f"⚠️ Retry {attempt + 1}: Could not parse JSON")
                continue

        except Exception as e:
            print(f"❌ API error (attempt {attempt + 1}): {e}")
            if attempt < max_retries:
                continue
            return json.dumps({"error": f"API call failed: {str(e)}"})

    return json.dumps({"error": "Could not get valid recommendation after retries"})


print("✅ Enhanced fashion recommendation loaded!")
print(f"   📚 {len(OCCASION_CATEGORIES)} occasion categories")
print(f"   👗 {len(DRESS_TYPE_RATING_RULES)} dress type rating rules")
print(f"   🔑 {len(OCCASION_KEYWORD_MAP)} occasion keywords")

✅ Enhanced fashion recommendation loaded!
   📚 26 occasion categories
   👗 12 dress type rating rules
   🔑 108 occasion keywords


In [ ]:
# ============================================
# CELL 11: CHATBOT — INTENT DETECTION (Updated)
# ============================================

def detect_intent(user_message):
    msg = user_message.lower().strip()

    fashion_keywords = [
        # Western clothing
        "dress", "outfit", "wear", "clothes", "clothing",
        "shirt", "top", "blouse", "pants", "jeans", "skirt",
        "shoes", "footwear", "heels", "sneakers", "boots",
        # Actions
        "recommend", "analyze", "suggest", "match", "matching", "suit",
        "look", "style", "fashion", "rate", "check",
        # Western occasions
        "wedding", "party", "interview", "date", "formal", "casual",
        "color", "colour", "fit", "fits",
        # Indian traditional clothing
        "saree", "sari", "saari", "lehenga", "kurta", "kurti",
        "sherwani", "salwar", "kameez", "anarkali", "sharara",
        "gharara", "dupatta", "dhoti", "mundu", "veshti",
        "churidar", "palazzo", "nehru jacket", "bandhgala",
        "pathani", "achkan", "chaniya choli", "ghaghra",
        "mekhela", "chador", "phiran", "lungi",
        # Indian occasions/festivals
        "diwali", "deepavali", "onam", "pongal", "eid",
        "navratri", "garba", "dandiya", "holi", "rakhi",
        "raksha bandhan", "ganesh chaturthi", "ganpati",
        "vishu", "ugadi", "baisakhi", "bihu", "lohri",
        "durga puja", "karva chauth", "sangeet", "mehndi",
        "haldi", "shaadi", "reception", "engagement",
        "festival", "traditional", "ethnic", "pooja", "puja",
        "temple", "function", "ceremony", "celebration",
        # Fabrics
        "silk", "cotton", "chiffon", "brocade", "linen",
        "kanjeevaram", "banarasi", "chanderi", "kasavu",
        # General
        "what to wear", "what should i wear", "dress code",
        "outfit idea", "help me dress"
    ]

    if any(kw in msg for kw in fashion_keywords):
        return _detect_fashion_intent_api(user_message)

    casual_patterns = [
        "hi", "hello", "hey", "hii", "hiii",
        "how are you", "how r u", "how are u",
        "what's up", "whats up", "sup",
        "good morning", "good evening", "good night",
        "thanks", "thank you", "thx",
        "ok", "okay", "cool", "nice", "great",
        "yes", "no", "yeah", "yep", "nope",
        "lol", "haha", "hehe",
        "i'm good", "im good", "i am good",
        "i'm fine", "im fine", "i am fine",
        "nothing much", "not much", "nm",
        "just chilling", "just bored", "bored", "can you hear me"
    ]

    if any(msg == p or msg.startswith(p + " ") or msg.endswith(" " + p)
           for p in casual_patterns):
        return {
            "is_fashion_request": False, "intent": None,
            "occasion": "none", "region": None, "is_off_topic": False
        }

    off_topic_patterns = [
        "math", "calculate", "2+2", "equation",
        "physics", "chemistry", "biology", "science",
        "president", "politics", "election", "government",
        "code", "programming", "python", "javascript",
        "weather", "temperature",
        "recipe", "cook", "food",
        "movie", "film", "song", "music"
    ]

    if any(p in msg for p in off_topic_patterns):
        return {
            "is_fashion_request": False, "intent": None,
            "occasion": "none", "region": None, "is_off_topic": True
        }

    return {
        "is_fashion_request": False, "intent": None,
        "occasion": "none", "region": None, "is_off_topic": False
    }


def _detect_fashion_intent_api(user_message):
    prompt = f"""Analyze this FASHION-related message. RESPOND ONLY WITH JSON.

Extract:
1. "intent": "basic" (want NEW outfit recommendation) | "matching" (CHECK/RATE current outfit) | "suggestion" (COMPLETE an existing outfit with missing pieces)
2. "occasion": Extract the specific occasion. Examples: "diwali", "onam", "wedding", "sangeet", "mehndi", "eid", "party", "interview", "date", "casual", "formal", "temple", "office", "college", "navratri", "holi", "christmas", "reception", "ganesh chaturthi" — or "none" if not mentioned
3. "region": ONLY for "suggestion" intent — "upper" (user HAS top/upper body clothing, needs bottom recommended) | "lower" (user HAS bottom/lower body clothing, needs top recommended) | null

REGION RULE — READ CAREFULLY:
- User is WEARING a SHIRT/TOP/BLOUSE/KURTA/SAREE BLOUSE/CHOLI → region = "upper" (they HAVE upper, need lower)
- User is WEARING PANTS/JEANS/SKIRT/SAREE/LEHENGA/CHURIDAR/MUNDU → region = "lower" (they HAVE lower, need upper)

EXAMPLES:
"recommend a dress for diwali" -> {{"intent":"basic","occasion":"diwali","region":null}}
"what should I wear to onam" -> {{"intent":"basic","occasion":"onam","region":null}}
"suggest an outfit for onam" -> {{"intent":"basic","occasion":"onam","region":null}}
"suggest a dress for onam" -> {{"intent":"basic","occasion":"onam","region":null}}
"suggest a dress for a funeral" -> {{"intent":"basic","occasion":"funeral","region":null}}
"outfit for my friend's sangeet" -> {{"intent":"basic","occasion":"sangeet","region":null}}
"what to wear to garba night" -> {{"intent":"basic","occasion":"navratri","region":null}}
"suggest an eid outfit" -> {{"intent":"basic","occasion":"eid","region":null}}
"dress for temple visit" -> {{"intent":"basic","occasion":"temple","region":null}}
"what should I wear to a party" -> {{"intent":"basic","occasion":"party","region":null}}
"recommend a casual outfit" -> {{"intent":"basic","occasion":"casual","region":null}}
"does this match?" -> {{"intent":"matching","occasion":"none","region":null}}
"rate my outfit" -> {{"intent":"matching","occasion":"none","region":null}}
"I am going for a wedding, rate my outfit" -> {{"intent":"matching","occasion":"wedding","region":null}}
"what pants go with my shirt?" -> {{"intent":"suggestion","occasion":"none","region":"upper"}}
"suggest a pant for my shirt" -> {{"intent":"suggestion","occasion":"none","region":"upper"}}
"suggest a pant based on this shirt" -> {{"intent":"suggestion","occasion":"none","region":"upper"}}
"suggest a pant for this shirt for a date night" -> {{"intent":"suggestion","occasion":"date night","region":"upper"}}
"I am going for a wedding, suggest a pant for this shirt" -> {{"intent":"suggestion","occasion":"wedding","region":"upper"}}
"suggest a lower body dress based on my upper body" -> {{"intent":"suggestion","occasion":"none","region":"upper"}}
"what should I pair with my saree blouse?" -> {{"intent":"suggestion","occasion":"none","region":"upper"}}
"what lehenga goes with this choli?" -> {{"intent":"suggestion","occasion":"none","region":"upper"}}
"suggest a saree for this blouse" -> {{"intent":"suggestion","occasion":"none","region":"upper"}}
"top for my pants?" -> {{"intent":"suggestion","occasion":"none","region":"lower"}}
"shirt for my pants?" -> {{"intent":"suggestion","occasion":"none","region":"lower"}}
"suggest a shirt for my pant" -> {{"intent":"suggestion","occasion":"none","region":"lower"}}
"suggest a shirt for this pant for a party" -> {{"intent":"suggestion","occasion":"party","region":"lower"}}
"suggest a shirt for my pants for a wedding?" -> {{"intent":"suggestion","occasion":"wedding","region":"lower"}}
"I am going for a party, suggest a shirt for this pant" -> {{"intent":"suggestion","occasion":"party","region":"lower"}}
"I am going for a wedding, suggest a top for this skirt" -> {{"intent":"suggestion","occasion":"wedding","region":"lower"}}
"suggest a blouse for my saree" -> {{"intent":"suggestion","occasion":"none","region":"lower"}}
"what top goes with my churidar?" -> {{"intent":"suggestion","occasion":"none","region":"lower"}}
"suggest a kurti for my palazzo" -> {{"intent":"suggestion","occasion":"none","region":"lower"}}
"suggest an upper body dress based on my lower body" -> {{"intent":"suggestion","occasion":"none","region":"lower"}}

MESSAGE: "{user_message}"
JSON:"""

    try:
        response = chatbot_client.chat.completions.create(
            model="deepseek/deepseek-chat",
            messages=[{"role": "user", "content": prompt}]
        )
        content = response.choices[0].message.content or ""
        parsed = _extract_json_from_text(content)
        if parsed:
            return {
                "is_fashion_request": True,
                "intent": parsed.get("intent"),
                "occasion": parsed.get("occasion", "none"),
                "region": parsed.get("region"),
                "is_off_topic": False
            }
    except Exception as e:
        print(f"❌ Intent API Error: {e}")

    return {
        "is_fashion_request": True, "intent": "basic",
        "occasion": "none", "region": None, "is_off_topic": False
    }


print("✅ Intent detection loaded!")

✅ Intent detection loaded!


In [ ]:
# ============================================
# CELL 12: CHATBOT — RESPONSE GENERATORS (Updated)
# ============================================

def generate_casual_response(user_message, conversation_history):
    recent_chat = ""
    if conversation_history:
        for msg in conversation_history[-4:]:
            role = "User" if msg["role"] == "user" else "Zyra"
            recent_chat += f"{role}: {msg['content']}\n"

    prompt = f"""You are Zyra, a friendly fashion assistant chatbot who knows both Western and Indian fashion.

RECENT CONVERSATION:
{recent_chat if recent_chat else "(Just started)"}

USER NOW SAYS: "{user_message}"

RULES:
1. Be friendly, warm, conversational
2. Respond naturally to greetings, small talk
3. After 1-2 casual exchanges, gently mention you can help with fashion
4. Keep responses as short points
5. NO markdown, NO asterisks
Your response:"""

    try:
        response = chatbot_client.chat.completions.create(
            model="deepseek/deepseek-chat",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content or "Hey! What's on your mind?"
    except Exception as e:
        print(f"❌ Casual response error: {e}")
        return "Hey there! I'm Zyra. Want to chat about fashion?"


def redirect_to_fashion(user_message):
    prompt = f"""You are Zyra, a friendly fashion assistant chatbot.
The user said something off-topic: "{user_message}"

Politely and playfully redirect to casual chat or fashion.
Keep it short (1-2 sentences). NO markdown.

Your response:"""

    try:
        response = chatbot_client.chat.completions.create(
            model="deepseek/deepseek-chat",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content or \
            "That's interesting! But I'm better at fashion. Want some outfit ideas?"
    except Exception as e:
        print(f"❌ Redirect response error: {e}")
        return "That's not really my area! But I'd love to help with fashion."


def explain_recommendation(intent, analysis, recommendation):
    dress_type = analysis.get('dress_type', 'unknown')
    dress_type_label = dress_type.replace('_', ' ').title() if dress_type != 'unknown' else ''

    if intent == "basic":
        if recommendation.get('outfit'):
            outfit_desc = f"- Outfit: {recommendation['outfit']}"
        else:
            parts = []
            if recommendation.get('upper'):
                parts.append(f"- Upper: {recommendation['upper']}")
            if recommendation.get('inner'):
                parts.append(f"- Inner: {recommendation['inner']}")
            if recommendation.get('lower'):
                parts.append(f"- Lower: {recommendation['lower']}")
            outfit_desc = "\n".join(parts) if parts else "- Outfit: Not specified"

        accessories_line = ""
        if recommendation.get('accessories'):
            accessories_line = f"\n- Accessories: {recommendation['accessories']}"

        context = (
            f"RECOMMENDATION:\n"
            f"{outfit_desc}\n"
            f"- Footwear: {recommendation.get('footwear', 'N/A')}"
            f"{accessories_line}\n"
            f"- Reason: {recommendation.get('reason', 'N/A')}\n\n"
            f"USER: Skin tone {analysis.get('skintone', 'unknown')}, {analysis.get('gender', 'unknown')}\n\n"
            f"Explain warmly. Mention each piece and why it works."
        )

    elif intent == "matching":
        rate = recommendation.get('rate', 'N/A')
        reason = recommendation.get('reason', 'N/A')
        tips = recommendation.get('tips', '')

        context = (
            f"RATING: {rate}/10\n"
            f"REASON: {reason}\n"
            f"IMPROVEMENT TIPS: {tips}\n\n"
            f"USER'S OUTFIT ({dress_type_label}):\n"
            f"- Upper: {analysis.get('upper', 'unknown')}\n"
            f"- Lower: {analysis.get('lower', 'unknown')}\n"
            f"- Footwear: {analysis.get('footwear', 'unknown')}\n"
            f"- Accessories: {analysis.get('accessories', 'none')}\n"
            f"- Fabric: {analysis.get('fabric_guess', 'unknown')}\n\n"
            f"Explain the rating naturally. Be specific about what works and what could improve. "
            f"If it's Indian traditional wear ({dress_type_label}), acknowledge the style appropriately."
        )

    elif intent == "suggestion":
        wearing = analysis.get('upper') or analysis.get('lower') or 'unknown'
        parts = []
        if recommendation.get('upper'):
            parts.append(f"- Upper: {recommendation['upper']}")
        if recommendation.get('lower'):
            parts.append(f"- Lower: {recommendation['lower']}")
        if recommendation.get('footwear'):
            parts.append(f"- Footwear: {recommendation['footwear']}")
        if recommendation.get('accessories'):
            parts.append(f"- Accessories: {recommendation['accessories']}")
        suggestion_lines = "\n".join(parts)

        context = (
            f"SUGGESTIONS:\n"
            f"{suggestion_lines}\n"
            f"- Reason: {recommendation.get('reason', 'N/A')}\n\n"
            f"USER IS WEARING: {wearing} (Dress type: {dress_type_label})\n"
            f"Fabric: {analysis.get('fabric_guess', 'unknown')}\n\n"
            f"Explain what to pair and why it works together. "
            f"If traditional wear, suggest traditional pairings."
        )
    else:
        return "I have some great fashion advice for you!"

    prompt = f"""{context}

RULES:
- Be friendly and enthusiastic
- ONLY explain what was recommended
- Keep it to 2-4 sentences
- NO markdown formatting
- Talk directly to the user
- If it's traditional Indian wear, show appreciation for the style
- Do not spell out emojis

Your explanation:"""

    try:
        response = chatbot_client.chat.completions.create(
           model="deepseek/deepseek-chat",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content or "Here's what I found for you!"
    except Exception as e:
        print(f"❌ Explain error: {e}")
        return "I have a great recommendation for you!"


print("✅ Response generators loaded!")

✅ Response generators loaded!


In [ ]:
# ============================================
# CELL 13: WEB PIPELINE — Updated for dress_type + enhanced analysis
# ============================================

def _capture_frame_from_b64(image_b64_string):
    """Convert a base64 image from browser into OpenCV BGR frame."""
    if ',' in image_b64_string:
        image_b64_string = image_b64_string.split(',')[1]
    binary = b64decode(image_b64_string)
    img_pil = Image.open(io.BytesIO(binary))
    img_rgb = np.array(img_pil)
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)


def run_fashion_pipeline_web(image_b64, intent, region=None, occasion=""):
    """Web version — accepts base64 image from browser."""
    global ui
    ui = ProfessionalUI()

    if intent not in ["basic", "matching", "suggestion"]:
        return {"success": False, "error": f"Invalid intent: {intent}"}
    if intent == "suggestion" and region not in ["upper", "lower"]:
        return {"success": False, "error": "For suggestion, region must be 'upper' or 'lower'"}

    config = DEFAULT_CONFIG.copy()
    ui.add_status("🚀", "Starting ZYRA Fashion AI")

    yolo_model = _yolo_model
    ui.add_status("✅", "Model ready")

    # Decode frame
    ui.add_status("📷", "Processing received frame...")
    try:
        captured_frame = _capture_frame_from_b64(image_b64)
    except Exception as e:
        return {"success": False, "error": f"Image decode failed: {str(e)}"}

    # Detect person
    ui.add_status("🔍", "Detecting person...")
    success, result = _detect_person(captured_frame, yolo_model, intent, region, config)

    if not success:
        return {"success": False, "error": result}

    # Bounding box preview
    display_frame = captured_frame.copy()
    x1, y1, x2, y2 = result['box']
    cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
    preview_b64 = _image_to_base64(display_frame)
    ui.add_status("✅", "Person detected!")

    # Crop
    h, w = captured_frame.shape[:2]
    crop_bbox = _get_crop_bbox(result['keypoints'], result['box'], result['crop_to'])
    cx1, cy1, cx2, cy2 = _expand_bbox(*crop_bbox, w, h)
    cropped = captured_frame[cy1:cy2, cx1:cx2]

    save_path = f"/tmp/zyra_captured_{intent}_{int(time.time())}.jpg"
    cv2.imwrite(save_path, cropped)
    ui.add_status("💾", f"Saved: {save_path}")

    # Gemini analysis
    ui.add_status("🤖", "Analyzing with Gemini AI...")
    img_pil = Image.open(save_path)
    gemini_response = analyze_image_with_gemini(img_pil, intent, region)
    analysis = parse_gemini_response(gemini_response)
    ui.add_status("✅", "Image analyzed")

    # Log detected dress type
    if analysis.get("dress_type"):
        ui.add_status("👗", f"Detected: {analysis['dress_type']}")

    # Build recommendation input with ALL new fields
    ui.add_status("👔", "Generating fashion recommendation...")
    fashion_input = {
        "intent": intent,
        "skintone": analysis.get("skintone", "Type 3"),
        "gender": analysis.get("gender", "Male"),
        "occasion": occasion if occasion else "none",
        "dress_type": analysis.get("dress_type", "unknown"),
        "fabric_guess": analysis.get("fabric_guess", "unknown"),
        "pattern": analysis.get("pattern", "unknown"),
    }

    if intent == "matching":
        fashion_input["upper"] = analysis.get("upper", "Not detected")
        fashion_input["lower"] = analysis.get("lower", "Not detected")
        fashion_input["footwear"] = analysis.get("footwear", "none")
        fashion_input["accessories"] = analysis.get("accessories", "none")
    elif intent == "suggestion":
        fashion_input["fabric_guess"] = analysis.get("fabric_guess", "unknown")
        fashion_input["pattern"] = analysis.get("pattern", "unknown")
        if region == "upper":
            fashion_input["upper"] = analysis.get("upper", "Not detected")
        elif region == "lower":
            fashion_input["lower"] = analysis.get("lower", "Not detected")

    recommendation_raw = get_fashion_recommendation(fashion_input)
    ui.add_status("✨", "Recommendation ready!")

    if isinstance(recommendation_raw, str):
        try:
            recommendation = json.loads(recommendation_raw)
        except (json.JSONDecodeError, ValueError):
            recommendation = {"raw": recommendation_raw}
    else:
        recommendation = recommendation_raw

    # Generate outfit image (for basic and suggestion only)
    generated_image = None
    if intent in ["basic", "suggestion"]:
        if not recommendation.get("error") and not recommendation.get("raw"):
            ui.add_status("🎨", "Generating outfit visualization...")
            generated_image = generate_outfit_image(recommendation, analysis)
            if generated_image["success"]:
                ui.add_status("✅", "Outfit image generated!")
            else:
                ui.add_status("⚠️", f"Image gen failed: {generated_image.get('error', 'Unknown')}")
        else:
            ui.add_status("⚠️", "Skipping image — recommendation not structured")

    return {
        "success": True,
        "preview_b64": preview_b64,
        "analysis": analysis,
        "recommendation": recommendation,
        "generated_image": generated_image,
        "intent": intent,
        "region": region,
        "occasion": occasion,
        "status_logs": ui.status_logs
    }


print("✅ Web pipeline loaded!")

✅ Web pipeline loaded!


In [ ]:
# ============================================
# CELL 14A: FLASK APP SETUP
# ============================================

app = Flask(__name__)
CORS(app)
app.secret_key = os.urandom(24).hex()
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 16 MB max

# Server-side session state (single-user Colab)
_state = {
    "is_active": False,
    "conversation_history": []
}

def _trim_history():
    if len(_state["conversation_history"]) > MAX_HISTORY:
        _state["conversation_history"] = _state["conversation_history"][-MAX_HISTORY:]

print("✅ Flask app created!")

✅ Flask app created!


In [ ]:
# ============================================
# CELL 14B: HTML TEMPLATE (Updated with dress_type, tips, accessories)
# ============================================

HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>ZYRA Fashion AI</title>
<link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=DM+Sans:wght@300;400;500&display=swap" rel="stylesheet">
<style>
*{box-sizing:border-box;margin:0;padding:0}
html,body{height:100%;overflow:hidden}
:root{
  --g:#00ffaa;--c:#00e5ff;--p:#a855f7;--pk:#ff2d9b;--gold:#ffd166;
  --fd:'Syne',sans-serif;--fb:'DM Sans',sans-serif;
}
body{background:#000;font-family:var(--fb);color:#fff;height:100vh;display:flex;flex-direction:column;overflow:hidden}

/* ================================================ SLEEP SCREEN */
#sleepScreen{
  position:fixed;inset:0;background:#000;
  display:flex;align-items:center;justify-content:center;
  z-index:1000;transition:opacity .4s ease,visibility .4s ease;
}
#sleepScreen.hidden{opacity:0;visibility:hidden;pointer-events:none}
.s-wrap{display:flex;flex-direction:column;align-items:center;gap:22px}
.s-ripple{position:relative;width:84px;height:84px;display:flex;align-items:center;justify-content:center}
.s-ripple::before,.s-ripple::after{
  content:'';position:absolute;inset:0;border-radius:50%;
  border:1px solid rgba(255,255,255,0.055);
  animation:sRip 3.2s ease-out infinite;
}
.s-ripple::after{animation-delay:1.6s;border-color:rgba(255,255,255,0.03)}
@keyframes sRip{0%{transform:scale(1);opacity:1}100%{transform:scale(2.8);opacity:0}}
.s-mic{
  position:relative;z-index:1;width:84px;height:84px;border-radius:50%;
  background:rgba(255,255,255,0.03);border:1px solid rgba(255,255,255,0.09);
  display:flex;align-items:center;justify-content:center;
  transition:all .5s ease;
}
.s-mic svg{width:32px;height:32px;opacity:0.22;transition:opacity .5s ease}
.s-mic.listening svg{opacity:1}
.s-mic.listening{
  background:rgba(0,255,170,0.07);border-color:rgba(0,255,170,0.28);
  box-shadow:0 0 40px rgba(0,255,170,0.12);
  animation:sBreath 1.5s ease-in-out infinite;
}
@keyframes sBreath{0%,100%{box-shadow:0 0 20px rgba(0,255,170,0.08)}50%{box-shadow:0 0 55px rgba(0,255,170,0.22)}}
.s-hint{font-size:10px;color:rgba(255,255,255,0.13);letter-spacing:3.5px;text-transform:uppercase;transition:color .5s ease}
.s-hint.active{color:rgba(0,255,170,0.45)}

/* ================================================ WAKEUP SCREEN */
#wakeupScreen{
  position:fixed;inset:0;background:#000;
  display:none;align-items:center;justify-content:center;
  z-index:999;overflow:hidden;
}
#wakeupScreen.active{display:flex}
#pCanvas{position:absolute;inset:0;pointer-events:none}

/* scan line */
.wu-scan{position:absolute;inset:0;pointer-events:none;overflow:hidden;opacity:0}
.wu-scan.go{opacity:1}
.wu-scan::after{
  content:'';position:absolute;left:0;right:0;height:1px;
  background:linear-gradient(90deg,transparent 0%,rgba(0,255,170,0.5) 20%,var(--g) 50%,rgba(0,255,170,0.5) 80%,transparent 100%);
  box-shadow:0 0 12px rgba(0,255,170,0.4),0 0 40px rgba(0,255,170,0.15);
  top:0;animation:scanSweep 1.1s cubic-bezier(.4,0,.6,1) forwards;
}
@keyframes scanSweep{0%{top:-2px}100%{top:100%}}

/* center content */
.wu-center{position:relative;z-index:2;display:flex;flex-direction:column;align-items:center}

/* icon with rotating rings */
.wu-icon-wrap{position:relative;width:130px;height:130px;display:flex;align-items:center;justify-content:center;margin-bottom:30px}
.wu-r1,.wu-r2,.wu-r3{position:absolute;border-radius:50%;border:1.5px solid transparent;opacity:0;transition:opacity .25s}
.wu-r1{inset:-20px;border-top-color:var(--g);border-right-color:rgba(0,255,170,0.25);animation:rr1 1.8s linear infinite}
.wu-r2{inset:-38px;border:1px solid transparent;border-bottom-color:var(--c);border-left-color:rgba(0,229,255,0.2);animation:rr2 2.6s linear infinite}
.wu-r3{inset:-56px;border:.5px solid transparent;border-top-color:var(--p);border-right-color:rgba(168,85,247,0.12);animation:rr1 3.4s linear infinite reverse}
#wakeupScreen.active .wu-r1,#wakeupScreen.active .wu-r2,#wakeupScreen.active .wu-r3{opacity:1}
@keyframes rr1{to{transform:rotate(360deg)}}
@keyframes rr2{to{transform:rotate(-360deg)}}

/* the icon box */
.wu-box{
  width:130px;height:130px;border-radius:36px;
  background:linear-gradient(145deg,rgba(0,255,170,0.13),rgba(0,229,255,0.06),rgba(168,85,247,0.09));
  border:1px solid rgba(0,255,170,0.22);
  display:flex;align-items:center;justify-content:center;font-size:54px;
  transform:scale(0);
  box-shadow:0 0 70px rgba(0,255,170,0.22),0 0 140px rgba(0,255,170,0.08),inset 0 1px 0 rgba(255,255,255,0.09);
}

/* LOGO text — clip-path reveal */
.wu-logo{
  font-family:var(--fd);font-size:76px;font-weight:800;letter-spacing:-3px;line-height:1;
  background:linear-gradient(135deg,#00ffaa 0%,#00e5ff 45%,#a855f7 100%);
  -webkit-background-clip:text;-webkit-text-fill-color:transparent;background-clip:text;
  clip-path:inset(0 100% 0 0);
}
.wu-sub{
  font-size:11px;letter-spacing:6px;text-transform:uppercase;
  color:rgba(255,255,255,0);margin-top:6px;
  transition:color .7s ease .7s;
}
.wu-sub.show{color:rgba(255,255,255,0.28)}

/* ================================================ MAIN APP */
.app{
  flex:1;display:flex;flex-direction:column;max-height:100%;
  opacity:0;transform:translateY(18px);
  transition:opacity .65s cubic-bezier(.4,0,.2,1),transform .65s cubic-bezier(.4,0,.2,1);
  position:relative;z-index:1;padding:14px;
  background:#060612;
}
.app::before{
  content:'';position:fixed;inset:0;
  background:
    radial-gradient(ellipse 60% 50% at 10% 15%,rgba(0,255,170,0.05) 0%,transparent 70%),
    radial-gradient(ellipse 50% 40% at 90% 80%,rgba(168,85,247,0.06) 0%,transparent 70%),
    radial-gradient(ellipse 40% 50% at 50% 50%,rgba(0,229,255,0.03) 0%,transparent 70%);
  pointer-events:none;z-index:0;animation:bgD 18s ease-in-out infinite alternate;
}
@keyframes bgD{0%{opacity:.5}100%{opacity:1}}
.app.visible{opacity:1;transform:translateY(0)}
.app.hidden{display:none}

/* HEADER */
.header{
  display:flex;justify-content:space-between;align-items:center;
  padding:0 2px 12px;margin-bottom:12px;
  border-bottom:1px solid rgba(255,255,255,0.07);flex-shrink:0;position:relative;z-index:1;
}
.header::after{content:'';position:absolute;bottom:-1px;left:0;width:100px;height:1px;background:linear-gradient(90deg,var(--g),transparent)}
.logo{display:flex;align-items:center;gap:10px}
.logo-icon{
  width:38px;height:38px;background:linear-gradient(135deg,rgba(0,255,170,0.14),rgba(0,229,255,0.08));
  border:1px solid rgba(0,255,170,0.25);border-radius:11px;
  display:flex;align-items:center;justify-content:center;font-size:18px;
  box-shadow:0 0 18px rgba(0,255,170,0.1),inset 0 1px 0 rgba(255,255,255,0.07);
}
.logo-text{font-family:var(--fd);font-size:20px;font-weight:800;letter-spacing:-.5px;background:linear-gradient(90deg,var(--g),var(--c));-webkit-background-clip:text;-webkit-text-fill-color:transparent;background-clip:text}
.logo-sub{font-size:9px;color:rgba(255,255,255,0.22);text-transform:uppercase;letter-spacing:2.5px;margin-top:-1px}
.hdr-r{display:flex;align-items:center;gap:8px}
.status-badge{background:rgba(0,255,170,0.07);border:1px solid rgba(0,255,170,0.22);padding:5px 12px;border-radius:20px;font-size:10px;color:var(--g);text-transform:uppercase;letter-spacing:.5px;font-weight:500}
.mic-ind{display:flex;align-items:center;gap:7px;background:rgba(0,0,0,0.4);border:1px solid rgba(255,255,255,0.07);border-radius:20px;padding:5px 10px}
.mic-dot{width:7px;height:7px;border-radius:50%;flex-shrink:0;transition:.4s}
.mic-dot.idle{background:#2a2a2a}
.mic-dot.hearing{background:var(--c);box-shadow:0 0 10px var(--c);animation:dp .6s ease-in-out infinite}
.mic-dot.active{background:var(--g);box-shadow:0 0 10px var(--g);animation:dp 1s ease-in-out infinite}
.mic-dot.muted{background:#ff4444}
@keyframes dp{0%,100%{opacity:1;transform:scale(1)}50%{opacity:.5;transform:scale(1.5)}}
.mic-lbl{font-size:10px;color:rgba(255,255,255,0.38);white-space:nowrap}
#muteBtn{background:rgba(255,255,255,0.04);border:1px solid rgba(255,255,255,0.08);color:rgba(255,255,255,0.38);border-radius:8px;padding:5px 10px;font-size:10px;cursor:pointer;transition:.2s;white-space:nowrap;font-family:var(--fb)}
#muteBtn:hover{background:rgba(255,255,255,0.08)}
#muteBtn.muted{background:rgba(255,68,68,0.1);border-color:rgba(255,68,68,0.28);color:#ff4444}
.speed-ctrl{display:flex;align-items:center;gap:6px;background:rgba(0,0,0,0.4);border:1px solid rgba(255,255,255,0.07);border-radius:20px;padding:5px 10px}
.speed-lbl{font-size:10px;color:rgba(255,255,255,0.38);white-space:nowrap}
.speed-val{font-size:10px;color:var(--g);font-weight:600;min-width:28px;text-align:right}
#speedSlider{-webkit-appearance:none;appearance:none;width:70px;height:3px;border-radius:2px;background:rgba(255,255,255,0.1);outline:none;cursor:pointer}
#speedSlider::-webkit-slider-thumb{-webkit-appearance:none;appearance:none;width:12px;height:12px;border-radius:50%;background:var(--g);box-shadow:0 0 6px rgba(0,255,170,0.5);cursor:pointer}
#speedSlider::-moz-range-thumb{width:12px;height:12px;border-radius:50%;background:var(--g);box-shadow:0 0 6px rgba(0,255,170,0.5);cursor:pointer;border:none}

/* DASHBOARD */
.dashboard{flex:1;display:grid;grid-template-columns:1fr 1.7fr;grid-template-rows:1fr auto;gap:12px;min-height:0;overflow:hidden;position:relative;z-index:1}
.dashboard.three-col{grid-template-columns:1fr 1fr 1.7fr}

/* Glass panels */
.gp{background:rgba(255,255,255,0.028);border:1px solid rgba(255,255,255,0.068);border-radius:16px;padding:12px;display:flex;flex-direction:column;min-height:0;position:relative;overflow:hidden;backdrop-filter:blur(14px);transition:border-color .3s,box-shadow .3s}
.gp::before{content:'';position:absolute;inset:0;border-radius:16px;background:linear-gradient(135deg,rgba(255,255,255,0.022) 0%,transparent 50%);pointer-events:none}
.gp:hover{border-color:rgba(255,255,255,0.11);box-shadow:0 8px 40px rgba(0,0,0,0.35)}

.ph{display:flex;align-items:center;gap:8px;margin-bottom:10px;padding-bottom:8px;border-bottom:1px solid rgba(255,255,255,0.05);flex-shrink:0}
.pi{width:26px;height:26px;border-radius:7px;display:flex;align-items:center;justify-content:center;font-size:12px}
.pi.g{background:rgba(0,255,170,0.09);border:1px solid rgba(0,255,170,0.16)}
.pi.c{background:rgba(0,229,255,0.09);border:1px solid rgba(0,229,255,0.16)}
.pi.p{background:rgba(168,85,247,0.09);border:1px solid rgba(168,85,247,0.16)}
.pt{font-size:9px;font-weight:500;color:rgba(255,255,255,0.38);text-transform:uppercase;letter-spacing:.8px}
.pa{margin-left:auto;width:6px;height:6px;border-radius:50%}
.pa.g{background:var(--g);box-shadow:0 0 7px var(--g);animation:dp 2s ease-in-out infinite}
.pa.c{background:var(--c);box-shadow:0 0 7px var(--c);animation:dp 2.5s ease-in-out infinite}

/* camera */
.cw{flex:1;background:rgba(0,0,0,0.5);border-radius:10px;display:flex;align-items:center;justify-content:center;position:relative;overflow:hidden;min-height:0;border:1px solid rgba(255,255,255,0.04)}
.cw::after{content:'';position:absolute;inset:0;border-radius:10px;background:linear-gradient(to bottom,rgba(0,255,170,0.018) 0%,transparent 30%,transparent 70%,rgba(0,0,0,0.22) 100%);pointer-events:none;z-index:1}
.cr{position:absolute;width:14px;height:14px;z-index:2}
.cr.tl{top:8px;left:8px;border-top:1.5px solid rgba(0,255,170,0.45);border-left:1.5px solid rgba(0,255,170,0.45)}
.cr.tr{top:8px;right:8px;border-top:1.5px solid rgba(0,255,170,0.45);border-right:1.5px solid rgba(0,255,170,0.45)}
.cr.bl{bottom:8px;left:8px;border-bottom:1.5px solid rgba(0,255,170,0.45);border-left:1.5px solid rgba(0,255,170,0.45)}
.cr.br{bottom:8px;right:8px;border-bottom:1.5px solid rgba(0,255,170,0.45);border-right:1.5px solid rgba(0,255,170,0.45)}

.gen-panel{background:rgba(255,255,255,0.028);border:1px solid rgba(255,255,255,0.068);border-radius:16px;padding:12px;display:none;flex-direction:column;min-height:0;backdrop-filter:blur(14px);position:relative;overflow:hidden}
.gen-panel.visible{display:flex}
.gen-panel::before{content:'';position:absolute;inset:0;border-radius:16px;background:linear-gradient(135deg,rgba(168,85,247,0.04),rgba(0,229,255,0.025));pointer-events:none}
.giw{flex:1;background:rgba(0,0,0,0.4);border-radius:10px;display:flex;align-items:center;justify-content:center;position:relative;overflow:hidden;min-height:0;border:1px solid rgba(255,255,255,0.04)}
.gen-outfit-img{max-width:100%;max-height:100%;object-fit:contain;border-radius:8px}
.img-loading{text-align:center;color:rgba(255,255,255,0.32);font-size:12px}
.img-loading .spinner{width:36px;height:36px;margin:0 auto 12px}
.img-error{text-align:center;color:#ff6b6b;font-size:12px;padding:20px}

#videoEl{max-width:100%;max-height:100%;object-fit:cover;border-radius:8px;display:none;position:relative;z-index:1;clip-path:inset(0 25% 0 25%)}
#resultImg{max-width:100%;max-height:100%;object-fit:contain;border-radius:8px;display:none;position:relative;z-index:1}
#snapCanvas{display:none}

.idle-ph{text-align:center;color:rgba(255,255,255,0.16);position:relative;z-index:1;display:flex;flex-direction:column;align-items:center}
.idle-ph .ii{font-size:34px;margin-bottom:10px;opacity:.35;animation:iF 3s ease-in-out infinite}
@keyframes iF{0%,100%{transform:translateY(0)}50%{transform:translateY(-6px)}}
.idle-ph .it{font-size:10px;color:rgba(255,255,255,0.16);letter-spacing:.4px}

.cdo{position:absolute;inset:0;background:rgba(0,0,0,0.15);display:none;flex-direction:column;align-items:center;justify-content:center;border-radius:10px;z-index:3}
.cdn{font-family:var(--fd);font-size:80px;font-weight:800;color:var(--g);line-height:1;text-shadow:0 0 40px rgba(0,255,170,0.45);animation:cdP .2s cubic-bezier(.34,1.56,.64,1)}
@keyframes cdP{0%{transform:scale(0.6)}100%{transform:scale(1)}}
.cdl{color:rgba(255,255,255,0.3);margin-top:8px;font-size:10px;letter-spacing:3px;text-transform:uppercase}

.det-badge{position:absolute;top:10px;left:50%;transform:translateX(-50%);padding:4px 14px;border-radius:20px;font-size:10px;font-weight:500;background:rgba(0,255,170,0.1);border:1px solid rgba(0,255,170,0.28);color:var(--g);display:none;z-index:2;letter-spacing:.5px}
.proc-ov{text-align:center;padding:12px;display:none}
.spinner{width:28px;height:28px;border:2px solid rgba(0,255,170,0.09);border-top-color:var(--g);border-radius:50%;animation:spin 1s linear infinite;margin:0 auto 8px}
@keyframes spin{to{transform:rotate(360deg)}}
.proc-msg{color:rgba(255,255,255,0.32);font-size:11px;letter-spacing:.4px}

/* right col */
.rc{display:flex;flex-direction:column;gap:12px;min-height:0;overflow:hidden;flex:1}

/* chat */
.chat-panel{flex:0 1 auto;min-height:120px;max-height:200px;background:rgba(255,255,255,0.028);border:1px solid rgba(255,255,255,0.068);border-radius:16px;padding:12px;display:flex;flex-direction:column;backdrop-filter:blur(14px);position:relative;overflow:hidden}
.chat-panel::before{content:'';position:absolute;top:0;left:0;right:0;height:1px;background:linear-gradient(90deg,transparent,rgba(0,229,255,0.22),transparent);pointer-events:none}
.chat-log{flex:1;overflow-y:auto;padding:6px;background:rgba(0,0,0,0.22);border-radius:10px;margin-bottom:8px;display:flex;flex-direction:column;gap:6px;scroll-behavior:smooth;min-height:0}
.chat-log::-webkit-scrollbar{width:3px}
.chat-log::-webkit-scrollbar-thumb{background:rgba(255,255,255,0.07);border-radius:10px}
.bubble{max-width:85%;padding:7px 11px;border-radius:12px;font-size:12px;line-height:1.5;animation:bIn .2s cubic-bezier(.34,1.56,.64,1)}
@keyframes bIn{0%{transform:scale(0.9) translateY(4px);opacity:0}100%{transform:scale(1) translateY(0);opacity:1}}
.bubble.user{background:linear-gradient(135deg,rgba(0,255,170,0.11),rgba(0,229,255,0.07));border:1px solid rgba(0,255,170,0.18);align-self:flex-end;color:rgba(255,255,255,0.86)}
.bubble.zyra{background:rgba(168,85,247,0.08);border:1px solid rgba(168,85,247,0.16);align-self:flex-start;color:rgba(255,255,255,0.8)}
.bubble.zyra .sender{font-size:8px;color:var(--p);margin-bottom:3px;font-weight:600;letter-spacing:1px;text-transform:uppercase}
.ci-row{display:flex;gap:6px;flex-shrink:0}
#chatInput{flex:1;background:rgba(0,0,0,0.28);border:1px solid rgba(255,255,255,0.07);border-radius:10px;padding:7px 12px;color:#fff;font-size:12px;outline:none;font-family:var(--fb);transition:border-color .2s,box-shadow .2s}
#chatInput:focus{border-color:rgba(0,255,170,0.32);box-shadow:0 0 0 2px rgba(0,255,170,0.055)}
#chatInput::placeholder{color:rgba(255,255,255,0.16)}
button{border:none;cursor:pointer;border-radius:10px;padding:7px 16px;font-size:12px;font-weight:600;transition:.2s;font-family:var(--fb)}
#sendBtn{background:linear-gradient(135deg,var(--g),#00cc88);color:#000;font-weight:700;box-shadow:0 4px 14px rgba(0,255,170,0.22)}
#sendBtn:hover{transform:translateY(-1px);box-shadow:0 6px 20px rgba(0,255,170,0.32)}
#sendBtn:active{transform:translateY(0)}

/* results */
.res-row{flex:1;gap:10px;min-height:0;overflow:auto;display:none}
.res-row.visible{display:grid;grid-template-columns:1fr 1.3fr}
.attr-card{background:linear-gradient(135deg,rgba(0,229,255,0.045),rgba(168,85,247,0.035));border:1px solid rgba(0,229,255,0.11);border-radius:14px;padding:14px;overflow-y:auto;position:relative}
.attr-card::before{content:'';position:absolute;top:0;left:0;right:0;height:1px;background:linear-gradient(90deg,transparent,rgba(0,229,255,0.32),transparent)}
.attr-card-title{font-family:var(--fd);color:var(--c);font-size:11px;font-weight:700;margin-bottom:12px;text-transform:uppercase;letter-spacing:1px;display:flex;align-items:center;gap:6px}
.rec-card{background:linear-gradient(135deg,rgba(0,255,170,0.045),rgba(0,229,255,0.025));border:1px solid rgba(0,255,170,0.14);border-radius:14px;padding:14px;overflow-y:auto;position:relative}
.rec-card::before{content:'';position:absolute;top:0;left:0;right:0;height:1px;background:linear-gradient(90deg,transparent,rgba(0,255,170,0.38),transparent)}
.rec-card-title{font-family:var(--fd);color:var(--g);font-size:11px;font-weight:700;margin-bottom:12px;text-transform:uppercase;letter-spacing:1px;display:flex;align-items:center;gap:6px}
.rec-row{display:flex;margin-bottom:9px;align-items:flex-start;gap:4px}
.rec-lbl{color:rgba(255,255,255,0.28);width:76px;font-size:10px;text-transform:uppercase;flex-shrink:0;font-weight:500;letter-spacing:.5px;padding-top:2px}
.rec-val{color:rgba(255,255,255,0.86);font-size:13px;flex:1;font-weight:400;line-height:1.3}
.rating-big{text-align:center;padding:8px}
.rating-num{font-family:var(--fd);font-size:54px;font-weight:800;line-height:1;letter-spacing:-2px}
.rating-reason{color:rgba(255,255,255,0.42);margin-top:8px;font-size:12px;line-height:1.5}
.tips-box{margin-top:12px;padding:10px 12px;background:rgba(0,255,170,0.045);border:1px solid rgba(0,255,170,0.11);border-radius:10px;text-align:left}
.tips-label{font-size:9px;color:var(--g);font-weight:600;margin-bottom:4px;text-transform:uppercase;letter-spacing:1px}
.tips-text{font-size:11px;color:rgba(255,255,255,0.42);line-height:1.5}
.attr-card::-webkit-scrollbar,.rec-card::-webkit-scrollbar{width:3px}
.attr-card::-webkit-scrollbar-thumb,.rec-card::-webkit-scrollbar-thumb{background:rgba(255,255,255,0.07);border-radius:10px}
</style>
</head>
<body>

<!-- ===== SLEEP: pure black + mic SVG ===== -->
<div id="sleepScreen">
  <div class="s-wrap">
    <div class="s-ripple">
      <div class="s-mic" id="sleepMic">
        <svg viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg">
          <rect x="9" y="2" width="6" height="11" rx="3" stroke="#00ffaa" stroke-width="1.5"/>
          <path d="M5 10a7 7 0 0014 0" stroke="#00ffaa" stroke-width="1.5" stroke-linecap="round"/>
          <line x1="12" y1="19" x2="12" y2="22" stroke="#00ffaa" stroke-width="1.5" stroke-linecap="round"/>
          <line x1="8" y1="22" x2="16" y2="22" stroke="#00ffaa" stroke-width="1.5" stroke-linecap="round"/>
        </svg>
      </div>
    </div>
    <div class="s-hint" id="sleepHint">Say "Zyra" to wake</div>
  </div>
</div>

<!-- ===== WAKEUP ANIMATION ===== -->
<div id="wakeupScreen">
  <canvas id="pCanvas"></canvas>
  <div class="wu-scan" id="wuScan"></div>

  <div class="wu-center">
    <div class="wu-icon-wrap">
      <div class="wu-r1"></div>
      <div class="wu-r2"></div>
      <div class="wu-r3"></div>
      <div class="wu-box" id="wuBox">👗</div>
    </div>
    <div class="wu-logo" id="wuLogo">ZYRA</div>
    <div class="wu-sub" id="wuSub">Fashion Intelligence</div>
  </div>
</div>

<!-- ===== MAIN APP ===== -->
<div class="app hidden" id="mainApp">
  <div class="header">
    <div class="logo">
      <div class="logo-icon">👗</div>
      <div>
        <div class="logo-text">ZYRA</div>
        <div class="logo-sub">Fashion Intelligence</div>
      </div>
    </div>
    <div class="hdr-r">
      <div class="mic-ind">
        <div class="mic-dot active" id="micDot"></div>
        <span class="mic-lbl" id="micLabel">Listening...</span>
      </div>
      <button id="muteBtn">🎤 Mute</button>
      <div class="speed-ctrl" style="display:none">
        <span class="speed-lbl">🗣 Speed</span>
        <input id="speedSlider" type="range" min="0.5" max="2.0" step="0.1" value="1.0"/>
        <span class="speed-val" id="speedVal">1.0×</span>
      </div>
      <div class="status-badge" id="statusBadge">✅ Active</div>
    </div>
  </div>

  <div class="dashboard" id="dashboard">
    <div class="gp">
      <div class="ph"><div class="pi g">📷</div><div class="pt">Live Camera</div><div class="pa g"></div></div>
      <div class="cw" id="cameraWrap">
        <div class="cr tl"></div><div class="cr tr"></div><div class="cr bl"></div><div class="cr br"></div>
        <div class="idle-ph" id="idlePlaceholder"><span class="ii">📸</span><div class="it">Camera activates when needed</div></div>
        <video id="videoEl" autoplay playsinline></video>
        <canvas id="snapCanvas"></canvas>
        <img id="resultImg"/>
        <div class="cdo" id="countdownOverlay"><div class="cdn" id="countdownNum">3</div><div class="cdl">Get Ready</div></div>
        <div class="det-badge" id="detectedBadge">✦ Detected</div>
      </div>
      <div class="proc-ov" id="processingOverlay"><div class="spinner"></div><div class="proc-msg">Analyzing with AI...</div></div>
    </div>

    <div class="gen-panel" id="generatedImagePanel">
      <div class="ph"><div class="pi p">🎨</div><div class="pt">AI Generated Outfit</div><div class="pa c"></div></div>
      <div class="giw" id="generatedImgWrap">
        <div class="img-loading" id="imgLoading"><div class="spinner"></div><div>Generating outfit visualization...</div></div>
        <img id="generatedOutfitImg" class="gen-outfit-img" style="display:none"/>
        <div class="img-error" id="imgError" style="display:none"></div>
      </div>
    </div>

    <div class="rc">
      <div class="chat-panel">
        <div class="ph"><div class="pi c">💬</div><div class="pt">Chat with Zyra</div><div class="pa c"></div></div>
        <div class="chat-log" id="chatLog"></div>
        <div class="ci-row">
          <input id="chatInput" type="text" placeholder="Ask about your style..." autocomplete="off"/>
        </div>
      </div>
      <div class="res-row" id="resultsRow">
        <div class="attr-card"><div class="attr-card-title"><span>📊</span> Detected</div><div id="attrRows"></div></div>
        <div class="rec-card"><div class="rec-card-title"><span>✨</span> Recommendation</div><div id="recRows"></div></div>
      </div>
    </div>
  </div>
</div>

<script>
/* STATE */
var mediaStream=null,isSpeaking=false,isCameraActive=false,isMuted=false;
var speechRate=1.3;
var isZyraActive=false,isUIVisible=false,speechEndTime=0,SPEECH_COOLDOWN=1500;
function isInSpeechCooldown(){return Date.now()-speechEndTime<SPEECH_COOLDOWN}
function shouldIgnoreMicInput(){return isSpeaking||isInSpeechCooldown()||isCameraActive}

/* PARTICLE SYSTEM */
var pCanvas=document.getElementById('pCanvas');
var pCtx=pCanvas.getContext('2d');
var parts=[];
function resizePCanvas(){pCanvas.width=window.innerWidth;pCanvas.height=window.innerHeight}
resizePCanvas();window.addEventListener('resize',resizePCanvas);

function spawnParts(){
  parts=[];
  var cx=pCanvas.width/2,cy=pCanvas.height/2;
  var cols=['#00ffaa','#00e5ff','#a855f7','#ff2d9b','#ffd166','#ffffff'];
  for(var i=0;i<90;i++){
    var a=Math.random()*Math.PI*2,spd=1.5+Math.random()*5.5,sz=.8+Math.random()*3;
    parts.push({x:cx,y:cy,vx:Math.cos(a)*spd,vy:Math.sin(a)*spd,life:1,dec:.007+Math.random()*0.014,sz:sz,col:cols[Math.floor(Math.random()*cols.length)],tr:[]});
  }
}

function hexAlpha(hex,a){var r=parseInt(hex.slice(1,3),16),g=parseInt(hex.slice(3,5),16),b=parseInt(hex.slice(5,7),16);return 'rgba('+r+','+g+','+b+','+a+')'}

function animParts(){
  if(!document.getElementById('wakeupScreen').classList.contains('active'))return;
  pCtx.clearRect(0,0,pCanvas.width,pCanvas.height);
  parts=parts.filter(function(p){return p.life>0});
  parts.forEach(function(p){
    p.tr.push({x:p.x,y:p.y});if(p.tr.length>8)p.tr.shift();
    p.x+=p.vx;p.y+=p.vy;p.vy+=0.04;p.life-=p.dec;
    for(var i=0;i<p.tr.length;i++){
      pCtx.globalAlpha=(i/p.tr.length)*p.life*0.28;
      pCtx.beginPath();pCtx.arc(p.tr[i].x,p.tr[i].y,p.sz*(i/p.tr.length)*0.5,0,Math.PI*2);
      pCtx.fillStyle=p.col;pCtx.fill();
    }
    pCtx.globalAlpha=p.life;
    pCtx.beginPath();pCtx.arc(p.x,p.y,p.sz,0,Math.PI*2);pCtx.fillStyle=p.col;pCtx.fill();
    pCtx.globalAlpha=1;
  });
  requestAnimationFrame(animParts);
}

/* WAKEUP ANIMATION SEQUENCE */
function playWakeup(done){
  var ws=document.getElementById('wakeupScreen');
  ws.classList.add('active');
  spawnParts();animParts();

  /* 1. icon box bounces in */
  var box=document.getElementById('wuBox');
  box.style.transition='transform .65s cubic-bezier(.34,1.56,.64,1), box-shadow .65s ease';
  box.style.transform='scale(1)';
  box.style.boxShadow='0 0 80px rgba(0,255,170,0.38),0 0 160px rgba(0,255,170,0.12),inset 0 1px 0 rgba(255,255,255,0.1)';

  /* 2. scan line fires */

  setTimeout(function(){

    var sc=document.getElementById('wuScan');

    sc.classList.add('go');
  },150);

  /* 3. logo text wipes in */
  setTimeout(function(){
    var logo=document.getElementById('wuLogo');
    logo.style.transition='clip-path .75s cubic-bezier(.4,0,.2,1)';
    logo.style.clipPath='inset(0 0% 0 0)';
  },350);

  /* 4. subtitle fades in */
  setTimeout(function(){
    document.getElementById('wuSub').classList.add('show');
  },750);

  /* 5. hold, then fade out, call done */
  setTimeout(function(){
    ws.style.transition='opacity .55s ease';
    ws.style.opacity='0';
    setTimeout(function(){
      ws.classList.remove('active');
      ws.style.opacity='';ws.style.transition='';
      /* reset for next time */
      box.style.transition='none';box.style.transform='scale(0)';box.style.boxShadow='';
      var l=document.getElementById('wuLogo');l.style.transition='none';l.style.clipPath='inset(0 100% 0 0)';
      document.getElementById('wuSub').classList.remove('show');
      document.getElementById('wuScan').classList.remove('go');
      if(done)done();
    },560);
  },2100);
}

/* SCREEN TRANSITIONS */
function showSleepScreen(){
  stopCamera();isCameraActive=false;
  document.getElementById('sleepMic').classList.remove('listening');
  document.getElementById('sleepHint').classList.remove('active');
  var ma=document.getElementById('mainApp');ma.classList.remove('visible');
  setTimeout(function(){ma.classList.add('hidden')},500);
  document.getElementById('sleepScreen').classList.remove('hidden');
  isUIVisible=false;isZyraActive=false;
  document.getElementById('chatLog').innerHTML='';
  document.getElementById('resultsRow').classList.remove('visible');
  document.getElementById('resultImg').style.display='none';
  document.getElementById('detectedBadge').style.display='none';
  document.getElementById('idlePlaceholder').style.display='flex';
  document.getElementById('generatedImagePanel').classList.remove('visible');
  document.getElementById('dashboard').classList.remove('three-col');
}

function showMainApp(){
  var ma=document.getElementById('mainApp');
  ma.classList.remove('hidden');
  setTimeout(function(){ma.classList.add('visible')},50);
  isUIVisible=true;isZyraActive=true;
  document.getElementById('statusBadge').textContent='✅ Active';
  setMicState('active','Listening...');
}

function triggerWakeUpAnimation(){
  /* fade out sleep */
  var ss=document.getElementById('sleepScreen');
  ss.style.transition='opacity .35s ease';ss.style.opacity='0';
  setTimeout(function(){ss.classList.add('hidden');ss.style.opacity='';ss.style.transition=''},350);
  /* play logo anim then go to app */
  playWakeup(function(){showMainApp()});
}

function setSleepMicListening(v){
  var sm=document.getElementById('sleepMic'),sh=document.getElementById('sleepHint');
  if(v){sm.classList.add('listening');sh.classList.add('active')}
  else{sm.classList.remove('listening');sh.classList.remove('active')}
}

function setMicState(s,l){var d=document.getElementById('micDot'),lb=document.getElementById('micLabel');if(d)d.className='mic-dot '+s;if(lb&&l)lb.textContent=l}

/* CHAT */
function addBubble(role,text){
  var log=document.getElementById('chatLog'),b=document.createElement('div');b.className='bubble '+role;
  if(role==='zyra'){var s=document.createElement('div');s.className='sender';s.textContent='ZYRA';b.appendChild(s);b.appendChild(document.createTextNode(text))}
  else{b.textContent=text}
  log.appendChild(b);log.scrollTop=log.scrollHeight;
}

function speakText(text,rate){
  rate=rate||speechRate;stopMicCompletely();isSpeaking=true;
  if(isUIVisible)setMicState('muted','Zyra speaking...');
  window.speechSynthesis.cancel();
  var sents=text.match(/[^.!?]+[.!?]+/g);if(!sents||!sents.length)sents=[text];
  var idx=0,voices=window.speechSynthesis.getVoices();
  var ev=voices.find(function(v){return v.lang.startsWith('en')});
  function next(){
    if(idx>=sents.length){handleSpeechEnd();return}
    var u=new SpeechSynthesisUtterance(sents[idx].trim());u.rate=rate;u.pitch=1;u.volume=1;u.lang='en-US';
    if(ev)u.voice=ev;u.onend=function(){idx++;next()};u.onerror=function(){idx++;next()};
    window.speechSynthesis.speak(u);
  }
  next();
}
function handleSpeechEnd(){
  isSpeaking=false;speechEndTime=Date.now();
  if(isUIVisible)setMicState('muted','Cooldown...');
  setTimeout(function(){if(!isMuted&&!isCameraActive&&!isSpeaking){resumeAlwaysOnMic();if(isUIVisible)setMicState('active','Listening...')}},SPEECH_COOLDOWN);
}

/* CAMERA */
async function startCamera(){
  try{mediaStream=await navigator.mediaDevices.getUserMedia({video:true});var v=document.getElementById('videoEl');v.srcObject=mediaStream;document.getElementById('idlePlaceholder').style.display='none';v.style.display='block'}
  catch(e){addBubble('zyra','Camera permission denied.')}
}
function stopCamera(){if(mediaStream){mediaStream.getTracks().forEach(function(t){t.stop()});mediaStream=null}document.getElementById('videoEl').style.display='none'}
function snapFrame(){
  var v=document.getElementById('videoEl'),c=document.getElementById('snapCanvas');
  var vw=v.videoWidth,vh=v.videoHeight;

  // Horizontal crop factor: 0.5 = keep center 50% width
  // Adjust: 0.6 = 60% width, 0.4 = 40% width (tighter)
  var horizontalCrop=0.5;

  var cropW=vw*horizontalCrop;
  var cropH=vh; // Full height preserved
  var cropX=(vw-cropW)/2; // Center horizontally
  var cropY=0; // Start from top

  c.width=cropW;
  c.height=cropH;
  c.getContext('2d').drawImage(v,cropX,cropY,cropW,cropH,0,0,cropW,cropH);
  return c.toDataURL('image/jpeg',0.95);
}
function runCountdown(sec){
  return new Promise(function(res){
    var ov=document.getElementById('countdownOverlay'),nm=document.getElementById('countdownNum');
    ov.style.display='flex';var ct=sec;nm.textContent=ct;
    var iv=setInterval(function(){ct--;if(ct<=0){clearInterval(iv);ov.style.display='none';res()}else{nm.textContent=ct;nm.style.animation='none';void nm.offsetWidth;nm.style.animation='cdP .2s cubic-bezier(.34,1.56,.64,1)'}},1000);
  });
}
async function doCaptureFlow(intent,region,occasion){
  var tm={basic:3,matching:5,suggestion:4},secs=tm[intent]||3;
  isCameraActive=true;stopMicCompletely();setMicState('muted','Camera active...');

  // === CLEAR ALL PREVIOUS RESULTS ===
  document.getElementById('resultImg').style.display='none';
  document.getElementById('resultImg').src='';
  document.getElementById('detectedBadge').style.display='none';
  document.getElementById('resultsRow').classList.remove('visible');
  document.getElementById('attrRows').innerHTML='';
  document.getElementById('recRows').innerHTML='';
  document.getElementById('generatedImagePanel').classList.remove('visible');
  document.getElementById('generatedOutfitImg').style.display='none';
  document.getElementById('generatedOutfitImg').src='';
  document.getElementById('imgError').style.display='none';
  document.getElementById('imgError').textContent='';
  document.getElementById('imgLoading').style.display='none';
  document.getElementById('dashboard').classList.remove('three-col');
  document.getElementById('idlePlaceholder').style.display='flex';
  // === END CLEAR ===

  var db=document.getElementById('dashboard'),ip=document.getElementById('generatedImagePanel');
  var il=document.getElementById('imgLoading'),ie=document.getElementById('generatedOutfitImg'),ir=document.getElementById('imgError');
  if(intent==='basic'||intent==='suggestion'){ip.classList.add('visible');db.classList.add('three-col');il.style.display='block';ie.style.display='none';ir.style.display='none'}
  else{ip.classList.remove('visible');db.classList.remove('three-col')}
  await startCamera();await new Promise(function(r){setTimeout(r,800)});await runCountdown(secs);
  var imgData=snapFrame();stopCamera();
  var ri=document.getElementById('resultImg');ri.src=imgData;ri.style.display='block';
  document.getElementById('idlePlaceholder').style.display='none';
  document.getElementById('processingOverlay').style.display='block';
  addBubble('zyra','Analyzing your photo...');
  try{
    var resp=await fetch('/api/process_frame',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({image:imgData,intent:intent,region:region,occasion:occasion})});
    var data=await resp.json();document.getElementById('processingOverlay').style.display='none';
    if(data.success){if(data.preview_b64){ri.src='data:image/jpeg;base64,'+data.preview_b64;document.getElementById('detectedBadge').style.display='block'}renderResults(data);var msg=data.chatbot_response||'Analysis complete!';addBubble('zyra',msg);speakText(msg)}
    else{var err=data.error||'Detection failed';addBubble('zyra',err);speakText(err+'. Please try again.');ip.classList.remove('visible');db.classList.remove('three-col')}
  }catch(e){document.getElementById('processingOverlay').style.display='none';addBubble('zyra','Connection error: '+e.message);ip.classList.remove('visible');db.classList.remove('three-col')}
  isCameraActive=false;
}
function renderResults(data){
  document.getElementById('resultsRow').classList.add('visible');
  var ar=document.getElementById('attrRows');ar.innerHTML='';
  var analysis=data.analysis||{};
  var attrOrder=['gender','skintone','dress_type','upper','lower','footwear','accessories','fabric_guess','pattern'];
  var attrLabels={gender:'Gender',skintone:'Skin Tone',dress_type:'Outfit Type',upper:'Upper',lower:'Lower',footwear:'Footwear',accessories:'Accessories',fabric_guess:'Fabric',pattern:'Pattern'};
  attrOrder.forEach(function(k){var val=analysis[k];if(val&&val!=='unknown'&&val!=='none'){var row=document.createElement('div');row.className='rec-row';var lbl=document.createElement('span');lbl.className='rec-lbl';lbl.textContent=(attrLabels[k]||k)+':';var vl=document.createElement('span');vl.className='rec-val';vl.textContent=val;row.appendChild(lbl);row.appendChild(vl);ar.appendChild(row)}});
  Object.keys(analysis).forEach(function(k){if(attrOrder.indexOf(k)===-1&&k!=='raw'&&k!=='error'){var val=analysis[k];if(val&&val!=='unknown'&&val!=='none'){var row=document.createElement('div');row.className='rec-row';var lbl=document.createElement('span');lbl.className='rec-lbl';lbl.textContent=k+':';var vl=document.createElement('span');vl.className='rec-val';vl.textContent=val;row.appendChild(lbl);row.appendChild(vl);ar.appendChild(row)}}});
  var rr=document.getElementById('recRows');rr.innerHTML='';
  var rec=data.recommendation||{};
  if(rec.rate!==undefined){
    var rd=document.createElement('div');rd.className='rating-big';
    var nd=document.createElement('div');nd.className='rating-num';nd.textContent=rec.rate+'/10';
    var rn=parseInt(rec.rate)||0;
    if(rn>=8)nd.style.color='#00ffaa';else if(rn>=6)nd.style.color='#ffd166';else if(rn>=4)nd.style.color='#ff8800';else nd.style.color='#ff4444';
    var rsd=document.createElement('div');rsd.className='rating-reason';rsd.textContent=rec.reason||'';
    rd.appendChild(nd);rd.appendChild(rsd);
    if(rec.tips){var tb=document.createElement('div');tb.className='tips-box';var tl=document.createElement('div');tl.className='tips-label';tl.textContent='💡 Tips to improve';var tt=document.createElement('div');tt.className='tips-text';tt.textContent=rec.tips;tb.appendChild(tl);tb.appendChild(tt);rd.appendChild(tb)}
    rr.appendChild(rd);
  }else{
    ['outfit','upper','inner','lower','footwear','accessories'].forEach(function(k){if(rec[k]){var row=document.createElement('div');row.className='rec-row';var lbl=document.createElement('span');lbl.className='rec-lbl';lbl.textContent=k+':';var val=document.createElement('span');val.className='rec-val';val.textContent=rec[k];row.appendChild(lbl);row.appendChild(val);rr.appendChild(row)}});
    if(rec.reason){var rRow=document.createElement('div');rRow.className='rec-row';rRow.style.cssText='margin-top:8px;border-top:1px solid rgba(255,255,255,0.06);padding-top:8px';var rl=document.createElement('span');rl.className='rec-lbl';rl.textContent='Why:';var rv=document.createElement('span');rv.className='rec-val';rv.style.cssText='font-size:12px;color:rgba(255,255,255,0.32)';rv.textContent=rec.reason;rRow.appendChild(rl);rRow.appendChild(rv);rr.appendChild(rRow)}
  }
  var gi=data.generated_image,db2=document.getElementById('dashboard'),ip2=document.getElementById('generatedImagePanel');
  var il2=document.getElementById('imgLoading'),ie2=document.getElementById('generatedOutfitImg'),ir2=document.getElementById('imgError');
  if(gi){ip2.classList.add('visible');db2.classList.add('three-col');if(gi.success){il2.style.display='none';ir2.style.display='none';ie2.src='data:image/jpeg;base64,'+gi.image_b64;ie2.style.display='block'}else{il2.style.display='none';ie2.style.display='none';ir2.textContent=gi.error||'Failed to generate image';ir2.style.display='block'}}
  else{ip2.classList.remove('visible');db2.classList.remove('three-col')}
}

async function sendMessage(msg){
  msg=msg.trim();if(!msg)return;if(isUIVisible)addBubble('user',msg);
  try{
    var resp=await fetch('/api/chat',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({message:msg})});
    var data=await resp.json();var reply=data.response||'';var lo=reply.toLowerCase();
    var isBye=lo.indexOf('bye')!==-1||lo.indexOf("say 'zyra'")!==-1||lo.indexOf('say "zyra"')!==-1;
    if(isUIVisible&&isBye&&data.action==='none'){addBubble('zyra',reply);speakText(reply);setTimeout(function(){showSleepScreen()},2500);return}
    if(isUIVisible){addBubble('zyra',reply);speakText(reply);document.getElementById('statusBadge').textContent='✅ Active';if(data.action==='capture'){setTimeout(async function(){await doCaptureFlow(data.intent,data.region||null,data.occasion||'')},1800)}}
  }catch(e){if(isUIVisible)addBubble('zyra','Server error: '+e.message)}
}

async function handleWakeUp(transcript){
  triggerWakeUpAnimation();
  await new Promise(function(r){setTimeout(r,1900)});
  try{
    var resp=await fetch('/api/chat',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({message:transcript})});
    var data=await resp.json();var reply=data.response||"Hey! I'm Zyra, your fashion buddy!";
    addBubble('zyra',reply);speakText(reply);
  }catch(e){var fb="Hey! I'm Zyra! What can I help you with?";addBubble('zyra',fb);speakText(fb)}
}

/* ALWAYS-ON MIC — all original logic preserved */
var alwaysOnRec=null,alwaysOnPaused=false,restartScheduled=false,restartDelay=150,MAX_RESTART_DELAY=5000;
var ZYRA_VARIATIONS=["zyra","zaira","zara","zyrah","zahra","zeera","zira","zyro","xara","syra","saira","zaara","zera","zyda","tyra"];
function initAlwaysOnMic(){
  var SR=window.SpeechRecognition||window.webkitSpeechRecognition;if(!SR){console.warn('No speech recognition');return}
  function startRec(){
    if(isMuted||alwaysOnPaused||alwaysOnRec)return;if(shouldIgnoreMicInput()){scheduleRestart(500);return}
    alwaysOnRec=new SR();alwaysOnRec.lang='en-US';alwaysOnRec.continuous=true;alwaysOnRec.interimResults=false;alwaysOnRec.maxAlternatives=1;
    alwaysOnRec.onstart=function(){restartScheduled=false;restartDelay=150;if(!isMuted&&!alwaysOnPaused&&isUIVisible&&!shouldIgnoreMicInput())setMicState('active','Listening...')};
    alwaysOnRec.onresult=function(e){
      if(shouldIgnoreMicInput())return;var r=e.results[e.results.length-1];if(!r.isFinal)return;
      var t=r[0].transcript.trim();if(!t)return;var lo=t.toLowerCase();
      if(!isZyraActive&&!isUIVisible){if(ZYRA_VARIATIONS.some(function(v){return lo.indexOf(v)!==-1})){setSleepMicListening(true);setTimeout(function(){handleWakeUp(t)},300)}}
      else if(isUIVisible){setMicState('hearing','Heard you!');sendMessage(t);setTimeout(function(){if(!isMuted&&!alwaysOnPaused&&isUIVisible&&!shouldIgnoreMicInput())setMicState('active','Listening...')},600)}
    };
    alwaysOnRec.onerror=function(e){if(e.error==='aborted')return;alwaysOnRec=null;restartDelay=Math.min(restartDelay*2,MAX_RESTART_DELAY);if(!shouldIgnoreMicInput())scheduleRestart(e.error==='no-speech'?100:restartDelay)};
    alwaysOnRec.onend=function(){alwaysOnRec=null;if(!isMuted&&!alwaysOnPaused&&!shouldIgnoreMicInput())scheduleRestart(restartDelay)};
    try{alwaysOnRec.start()}catch(e){alwaysOnRec=null;scheduleRestart(1000)}
  }
  function scheduleRestart(d){
    if(restartScheduled||isMuted||alwaysOnPaused)return;
    if(shouldIgnoreMicInput()){setTimeout(function(){if(!shouldIgnoreMicInput())startRec()},d);return}
    restartScheduled=true;setTimeout(function(){restartScheduled=false;startRec()},d);
  }
  window._micStart=startRec;startRec();
}
function stopMicCompletely(){alwaysOnPaused=true;if(alwaysOnRec){try{alwaysOnRec.abort()}catch(e){}alwaysOnRec=null}}
function resumeAlwaysOnMic(){if(isMuted||isCameraActive||isSpeaking||isInSpeechCooldown())return;alwaysOnPaused=false;if(window._micStart)window._micStart()}

document.getElementById('muteBtn').addEventListener('click',function(){
  var btn=document.getElementById('muteBtn');isMuted=!isMuted;
  if(isMuted){stopMicCompletely();btn.textContent='🔇 Unmute';btn.classList.add('muted');setMicState('muted','Muted')}
  else{btn.textContent='🎤 Mute';btn.classList.remove('muted');alwaysOnPaused=false;if(!shouldIgnoreMicInput()&&window._micStart)window._micStart()}
});
document.getElementById('chatInput').addEventListener('keydown',function(e){if(e.key==='Enter'){sendMessage(e.target.value);e.target.value=''}});
window.speechSynthesis.getVoices();
window.speechSynthesis.onvoiceschanged=function(){window.speechSynthesis.getVoices()};
document.getElementById('speedSlider').addEventListener('input',function(){
  speechRate=parseFloat(this.value);
  document.getElementById('speedVal').textContent=speechRate.toFixed(1)+'×';
});
initAlwaysOnMic();
</script>
</body>
</html>"""

print("✅ HTML template loaded! Length:", len(HTML_TEMPLATE))

✅ HTML template loaded! Length: 43849


In [ ]:
# ============================================
# CELL 14C: FLASK ROUTES
# ============================================

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)


@app.route('/api/chat', methods=['POST'])
def api_chat():
    data = flask_request.get_json(force=True)
    user_message = data.get('message', '').strip()
    if not user_message:
        return jsonify({"error": "No message provided"}), 400

    # Activation check
    if not _state["is_active"]:
        zyra_variations = ["zyra", "zaira", "zara", "zyrah", "zahra", "zeera", "zira",
                           "zyro", "xara", "syra", "saira", "zaara", "zera", "zyda", "tyra"]
        if any(name in user_message.lower() for name in zyra_variations):
            _state["is_active"] = True
            greeting = ("Hey! I'm Zyra, your fashion buddy! "
                        "I can recommend outfits, rate what you're wearing, "
                        "or suggest pieces to complete your look. "
                        "I'll use your camera to analyze your style! What's up?")
            _state["conversation_history"].append({"role": "assistant", "content": greeting})
            _trim_history()
            return jsonify({"response": greeting, "action": "none"})
        return jsonify({"response": "Say 'Zyra' to wake me up!", "action": "none"})

    # Deactivation
    if any(w in user_message.lower() for w in ["bye", "goodbye", "exit", "quit"]):
        _state["is_active"] = False
        _state["conversation_history"] = []
        return jsonify({"response": "Bye! Say 'Zyra' whenever you want to talk!", "action": "none"})

    # Detect intent
    intent_data = detect_intent(user_message)

    if intent_data.get("is_off_topic"):
        response = redirect_to_fashion(user_message)
        _state["conversation_history"].append({"role": "user", "content": user_message})
        _state["conversation_history"].append({"role": "assistant", "content": response})
        _trim_history()
        return jsonify({"response": response, "action": "none"})

    if not intent_data.get("is_fashion_request"):
        response = generate_casual_response(user_message, _state["conversation_history"])
        _state["conversation_history"].append({"role": "user", "content": user_message})
        _state["conversation_history"].append({"role": "assistant", "content": response})
        _trim_history()
        return jsonify({"response": response, "action": "none"})

    # Fashion request
    intent = intent_data.get("intent")
    occasion = intent_data.get("occasion", "none")
    region = intent_data.get("region")
    occasion_str = "" if occasion in ["none", None, ""] else occasion

    pre_msg = {
        "basic": "Let me see you! Get ready for a quick photo so I can analyze your skin tone!",
        "matching": "Show me your full outfit! I'll rate how everything looks together!",
        "suggestion": "Let me see what you're wearing! I'll suggest pieces to complete your look!"
    }.get(intent, "Get ready for the camera!")

    _state["conversation_history"].append({"role": "user", "content": user_message})
    _trim_history()

    return jsonify({
        "response": pre_msg,
        "action": "capture",
        "intent": intent,
        "occasion": occasion_str,
        "region": region
    })


@app.route('/api/process_frame', methods=['POST'])
def api_process_frame():
    data = flask_request.get_json(force=True)
    image_b64 = data.get('image')
    intent = data.get('intent', 'basic')
    region = data.get('region')
    occasion = data.get('occasion', '')

    if not image_b64:
        return jsonify({"success": False, "error": "No image provided"}), 400

    result = run_fashion_pipeline_web(image_b64, intent, region, occasion)

    if result["success"]:
        chatbot_response = explain_recommendation(
            intent, result["analysis"], result["recommendation"]
        )
        result["chatbot_response"] = chatbot_response
        _state["conversation_history"].append({"role": "assistant", "content": chatbot_response})
        _trim_history()

    return jsonify(result)


@app.route('/api/status', methods=['GET'])
def api_status():
    return jsonify({"active": _state["is_active"]})


@app.route('/api/reset', methods=['POST'])
def api_reset():
    _state["is_active"] = False
    _state["conversation_history"] = []
    return jsonify({"success": True})


print("✅ Flask routes loaded!")

✅ Flask routes loaded!


In [ ]:
# ============================================
# CELL 15: START SERVER
# ============================================
import getpass

def start_webapp(port=5000, ngrok_auth_token=None):
    token = ngrok_auth_token or os.environ.get("NGROK_AUTHTOKEN")
    if token:
        pyngrok_client.set_auth_token(token)

    tunnel = pyngrok_client.connect(port, bind_tls=True)
    public_url = tunnel.public_url

    print("\n" + "=" * 60)
    print("  🌟  ZYRA Fashion AI — Running!")
    print("=" * 60)
    print(f"  🔗  Public URL : {public_url}")
    print(f"  🏠  Local URL  : http://localhost:{port}")
    print("=" * 60)
    print("  Open the Public URL in your browser.")
    print("  Press Stop to shut down.\n")

    flask_thread = threading.Thread(
        target=lambda: app.run(host="0.0.0.0", port=port, debug=False, use_reloader=False)
    )
    flask_thread.daemon = True
    flask_thread.start()
    print("  Flask thread started.\n")


NGROK_TOKEN = getpass.getpass("🔑 Paste your ngrok auth token: ")
PORT = 5000
start_webapp(port=PORT, ngrok_auth_token=NGROK_TOKEN)

print("\n✅ ZYRA is live! Open the Public URL above in your browser.")
print("   This cell must stay running. Stop it to shut down the server.\n")
while True:
    time.sleep(60)


  🌟  ZYRA Fashion AI — Running!
  🔗  Public URL : https://unobeyed-adelaida-picked.ngrok-free.dev
  🏠  Local URL  : http://localhost:5000
  Open the Public URL in your browser.
  Press Stop to shut down.

  Flask thread started.


✅ ZYRA is live! Open the Public URL above in your browser.
   This cell must stay running. Stop it to shut down the server.

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [05/May/2026 10:33:55] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 10:34:19] "POST /api/chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 10:34:31] "POST /api/chat HTTP/1.1" 200 -


  🚀 Starting ZYRA Fashion AI
  ✅ Model ready
  📷 Processing received frame...
  🔍 Detecting person...

0: 640x448 2 persons, 826.6ms
Speed: 17.0ms preprocess, 826.6ms inference, 4.6ms postprocess per image at shape (1, 3, 640, 448)
  ✅ Person detected!
  💾 Saved: /tmp/zyra_captured_basic_1777977278.jpg
  🤖 Analyzing with Gemini AI...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6555.33ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2504.58ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2908.38ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4300.49ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3921.65ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4350.84ms


❌ Gemini error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 50.816547253s.
⚠️ Gemini parse error: Invalid control character at: line 1 column 385 (char 384)
  ✅ Image analyzed
  👔 Generating fashion recommendation...
  ✨ Recommendation ready!
  🎨 Generating outfit visualization...
🎨 Generating image with prompt: Professional fashion photography of a person fashion model with light to medium skin tone, wearing Light saffron cotton ...
✅ Image generated: https://v3b.fal.media/files/b/0a98fa63/jgGRzFvfEEDg5rUR6otHw.jp

INFO:werkzeug:127.0.0.1 - - [05/May/2026 10:35:19] "POST /api/process_frame HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 10:35:51] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 10:36:11] "POST /api/chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 10:36:23] "POST /api/chat HTTP/1.1" 200 -


  🚀 Starting ZYRA Fashion AI
  ✅ Model ready
  📷 Processing received frame...
  🔍 Detecting person...

0: 640x448 1 person, 750.7ms
Speed: 6.0ms preprocess, 750.7ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 448)
  ✅ Person detected!
  💾 Saved: /tmp/zyra_captured_basic_1777977391.jpg
  🤖 Analyzing with Gemini AI...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4933.05ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3567.05ms


❌ Gemini error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 17.995623595s.
⚠️ Gemini parse error: Invalid control character at: line 1 column 385 (char 384)
  ✅ Image analyzed
  👔 Generating fashion recommendation...
  ✨ Recommendation ready!
  🎨 Generating outfit visualization...
🎨 Generating image with prompt: Professional fashion photography of a person fashion model with light to medium skin tone, wearing Simple cotton kurta i...
✅ Image generated: https://v3b.fal.media/files/b/0a98fa6c/S8VixHBVKdGCEpav2YudC.j

INFO:werkzeug:127.0.0.1 - - [05/May/2026 10:36:53] "POST /api/process_frame HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 10:37:31] "POST /api/chat HTTP/1.1" 200 -
